version 1 de codigo de deeplearning echo por claude

In [ ]:
"""
================================================================================
  SURROGATE MODEL — CALIDAD DEL AIRE  |  Interfaz Web Gradio
  Autor   : Arquitecto de Datos / MLOps Engineer
  Versión : 6.0  (Deep Learning Multi-Salida — MLP Keras/TensorFlow)
  Re-arquitectura completa desde v5.0 (Gradient Boosting → Red Neuronal)

  Requerimientos MLOps aplicados:
      [DL-1]  Detección dinámica de esquema X/Y (exact string match)
      [DL-2]  Imputación temporal síncrona (ffill → bfill → mean)
      [DL-3]  MLP Multi-Salida nativa (6 neuronas output, CUDA/V100)
      [DL-4]  Curva de aprendizaje global única (Train Loss / Val Loss)
      [DL-5]  Inferencia limpia (vars climáticas → 6 contaminantes)
================================================================================
  Dependencias:
      pip install gradio scikit-learn pandas numpy matplotlib seaborn tensorflow
================================================================================
"""

# ──────────────────────────────────────────────────────────────────────────────
# 0.  CONFIGURACIÓN MLOPS CORE & IMPORTACIONES (ZONA CRÍTICA DE INFRAESTRUCTURA)
# ──────────────────────────────────────────────────────────────────────────────
import os
import sys

# ── INYECCIÓN SISTEMA OPERATIVO: Evita el acaparamiento y bloqueos de contexto CUDA ──
os.environ['KERAS_BACKEND'] = 'tensorflow'
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

import warnings
import logging
import traceback
import subprocess
import pickle
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import gradio as gr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ── TensorFlow / Keras GPU Initialization ─────────────────────────────────────
try:
    import tensorflow as tf
    TF_DISPONIBLE = True

    # Forzar el crecimiento dinámico inmediatamente antes de cargar Keras layers
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✅ MLOPS: Crecimiento dinámico de memoria inyectado con éxito en la V100.")
        TF_GPU_MSG = f"TensorFlow GPU: {len(gpus)} dispositivo(s) activo(s) [memory_growth=True]"
    else:
        TF_GPU_MSG = "TensorFlow: modo CPU (no se detectó GPU)"
    
    from tensorflow import keras
    log.info(TF_GPU_MSG)

except Exception as e:
    TF_DISPONIBLE = False
    TF_GPU_MSG = f"TensorFlow no inicializado correctamente: {e}"
    log.warning(TF_GPU_MSG)

# ──────────────────────────────────────────────────────────────────────────────
# 1.  CONSTANTES Y ESTADO DE SESIÓN
# ──────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("resultados_surrogate")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── [DL-1]  Lista exacta de contaminantes Y (REGLA DE ORO — no modificar) ─────
CONTAMINANTES_Y: list[str] = ["PM25", "PM10", "O3", "CO", "NO2", "SO2"]

# Columnas de control excluidas tanto de X como de Y
COLUMNAS_CONTROL: set[str] = {
    "Fecha", "fecha", "FECHA",
    "Parroquia", "parroquia", "PARROQUIA",
    "Station", "station", "STATION",
    "Estacion", "estacion",
}

ANOS_PANDEMIA: list[int] = [2020, 2021]

# Variables climáticas base (UI de inferencia)
VARS_CLIMATICAS_BASE: list[str] = [
    "Temperatura", "Humedad",
    "Viento_Velocidad", "Viento_Direccion",
    "Precipitacion",
]

# Estado global de sesión (un modelo por CSV cargado)
SESION: dict = {
    "modelo":       None,          # keras.Model en memoria
    "scaler_x":     None,          # StandardScaler ajustado con datos de train
    "scaler_y":     None,          # StandardScaler para Y (mejora convergencia)
    "feat_cols":    [],            # lista de columnas X
    "target_cols":  [],            # subset de CONTAMINANTES_Y presentes en el CSV
    "parroquia":    "",
    "last_row":     {},            # último registro X para inyección de lags
    "ultimo_mes":   datetime.now().month,
    "ruta_modelo":  "",
}

# Paleta de colores
COLOR_TRAIN = "#3B82F6"    # azul  — train
COLOR_VAL   = "#F97316"    # naranja — validación
COLORES_6   = ["#22C55E", "#3B82F6", "#F97316", "#A855F7", "#EC4899", "#14B8A6"]
BG_PLOT     = "#0F172A"
TEXT_PLOT   = "#E2E8F0"
GRID_PLOT   = "#1E293B"


# ──────────────────────────────────────────────────────────────────────────────
# 2.  DETECCIÓN DE GPU
# ──────────────────────────────────────────────────────────────────────────────
def detectar_gpu() -> tuple[bool, str]:
    # Vía 1 — nvidia-smi
    try:
        res = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=8,
        )
        if res.returncode == 0 and res.stdout.strip():
            nombre = res.stdout.strip().split("\n")[0]
            return True, f"GPU detectada (nvidia-smi): {nombre}"
    except Exception:
        pass
    # Vía 2 — TensorFlow
    if TF_DISPONIBLE and tf.config.list_physical_devices("GPU"):
        return True, TF_GPU_MSG
    # Vía 3 — PyTorch (solo detección)
    try:
        import torch
        if torch.cuda.is_available():
            return True, f"GPU detectada (torch): {torch.cuda.get_device_name(0)}"
    except ImportError:
        pass
    return False, "CPU mode — no se detectó GPU"

GPU_DISPONIBLE, GPU_MSG = detectar_gpu()
log.info(GPU_MSG)


# ──────────────────────────────────────────────────────────────────────────────
# 3.  UTILIDADES CSV
# ──────────────────────────────────────────────────────────────────────────────
def listar_csvs(directorio: str = ".") -> list[str]:
    csvs = []
    for raiz, _, archivos in os.walk(directorio):
        for f in archivos:
            if f.lower().endswith(".csv"):
                csvs.append(os.path.relpath(os.path.join(raiz, f), directorio))
    csvs.sort()
    return csvs if csvs else ["(No se encontraron archivos CSV)"]


def refrescar_dropdown() -> gr.Dropdown:
    opciones = listar_csvs()
    return gr.Dropdown(choices=opciones, value=opciones[0] if opciones else None)


def _cargar_csv(ruta: str) -> pd.DataFrame:
    """[FIX-1] Ignora cabeceras decorativas con '#'."""
    try:
        return pd.read_csv(ruta, comment="#", low_memory=False, on_bad_lines="warn")
    except Exception as e:
        raise RuntimeError(f"Error al leer '{ruta}': {e}") from e


def _detectar_timestamp(df: pd.DataFrame) -> str | None:
    keywords = ("time", "fecha", "date", "hora", "datetime", "timestamp")
    candidatos = [c for c in df.columns if any(k in c.lower() for k in keywords)]
    if candidatos:
        return candidatos[0]
    for c in df.columns:
        try:
            pd.to_datetime(df[c].dropna().astype(str).iloc[:10], infer_datetime_format=True)
            return c
        except Exception:
            continue
    return None


# ──────────────────────────────────────────────────────────────────────────────
# 4.  [DL-1]  DETECCIÓN DINÁMICA DE ESQUEMA X / Y
# ──────────────────────────────────────────────────────────────────────────────
def _detectar_columnas_xy(
    df: pd.DataFrame,
    ts_col: str | None,
) -> tuple[list[str], list[str]]:
    """
    Separa el dataset en matrices X e Y aplicando coincidencia exacta de strings.

    Y (Matriz Objetivo):
        Contiene ÚNICAMENTE columnas cuyo nombre coincida de forma exacta
        con algún elemento de CONTAMINANTES_Y = ['PM25','PM10','O3','CO','NO2','SO2'].
        Regla de Oro: ninguna variable ajena a esa lista puede entrar en Y.

    X (Matriz de Entrada):
        Todas las columnas numéricas restantes, excluyendo:
            - Las columnas de Y
            - Las columnas de control (COLUMNAS_CONTROL)
            - La columna de timestamp (ts_col)
    """
    cols_set = set(df.columns)

    # Y: exact match
    y_cols: list[str] = [c for c in CONTAMINANTES_Y if c in cols_set]

    # Exclusiones para X
    excluir: set[str] = (
        set(y_cols) | COLUMNAS_CONTROL | ({ts_col} if ts_col else set())
    )

    # X: columnas numéricas que no están en la lista de exclusión
    x_cols: list[str] = []
    seen: set[str] = set()
    for c in df.columns:
        if c in excluir or c in seen:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            x_cols.append(c)
            seen.add(c)

    return x_cols, y_cols


def obtener_info_esquema(ruta: str) -> str:
    """Devuelve un resumen del esquema X/Y detectado para mostrar en la UI."""
    if not ruta or ruta.startswith("(No"):
        return "⚠️ Selecciona un archivo válido."
    try:
        df_head = pd.read_csv(ruta, comment="#", nrows=5, low_memory=False)
        ts = _detectar_timestamp(df_head)
        x_cols, y_cols = _detectar_columnas_xy(df_head, ts)
        return (
            f"✅ **Timestamp:** `{ts}`\n\n"
            f"🎯 **Y — Contaminantes ({len(y_cols)}):** `{y_cols}`\n\n"
            f"📥 **X — Features ({len(x_cols)}):** {x_cols[:8]}"
            f"{'…' if len(x_cols) > 8 else ''}"
        )
    except Exception as e:
        return f"❌ Error al inspeccionar esquema: {e}"


# ──────────────────────────────────────────────────────────────────────────────
# 5.  [DL-2]  IMPUTACIÓN TEMPORAL SÍNCRONA (Anti-NaN)
# ──────────────────────────────────────────────────────────────────────────────
def _imputar_temporal(
    df_X: pd.DataFrame,
    df_Y: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Aplica Forward-Fill → Backward-Fill → relleno por media de columna.
    Garantiza que la Red Neuronal reciba tensores numéricos limpios y completos.
    Opera de forma síncrona sobre ambas matrices para mantener el índice alineado.
    """
    df_X = df_X.ffill().bfill().fillna(df_X.mean(numeric_only=True))
    df_Y = df_Y.ffill().bfill().fillna(df_Y.mean(numeric_only=True))
    # Última línea de defensa: reemplazar NaN residuales con 0
    df_X = df_X.fillna(0.0)
    df_Y = df_Y.fillna(0.0)
    return df_X, df_Y


# ──────────────────────────────────────────────────────────────────────────────
# 6.  [DL-3]  ARQUITECTURA DE RED NEURONAL TABULAR MULTI-SALIDA
# ──────────────────────────────────────────────────────────────────────────────
def _construir_red_neuronal(n_features: int, n_outputs: int) -> "keras.Model":
    """
    MLP Tabular Multi-Salida para regresión.
    Optimizado para GPU Tesla V100 (BatchNorm + Dropout calibrado).

    Arquitectura:
        Input(n_features)
        → Dense(256) → BatchNormalization → ReLU → Dropout(0.30)
        → Dense(128) → BatchNormalization → ReLU → Dropout(0.20)
        → Dense(64)  → BatchNormalization → ReLU → Dropout(0.10)
        → Dense(n_outputs, activación lineal)   ← 6 neuronas de salida

    Compilación:
        Optimizador : Adam (lr=1e-3)
        Pérdida     : MSE (conjunto, multi-salida)
        Métrica     : MAE (para monitoreo visual)
    """
    if not TF_DISPONIBLE:
        raise ImportError(
            "TensorFlow no está instalado.\nEjecuta: pip install tensorflow"
        )

    inputs = keras.Input(shape=(n_features,), name="features_entrada")

    # Bloque 1
    x = keras.layers.Dense(256, use_bias=False, name="dense_256")(inputs)
    x = keras.layers.BatchNormalization(name="bn_1")(x)
    x = keras.layers.Activation("relu", name="relu_1")(x)
    x = keras.layers.Dropout(0.30, name="dropout_30")(x)

    # Bloque 2
    x = keras.layers.Dense(128, use_bias=False, name="dense_128")(x)
    x = keras.layers.BatchNormalization(name="bn_2")(x)
    x = keras.layers.Activation("relu", name="relu_2")(x)
    x = keras.layers.Dropout(0.20, name="dropout_20")(x)

    # Bloque 3
    x = keras.layers.Dense(64, use_bias=False, name="dense_64")(x)
    x = keras.layers.BatchNormalization(name="bn_3")(x)
    x = keras.layers.Activation("relu", name="relu_3")(x)
    x = keras.layers.Dropout(0.10, name="dropout_10")(x)

    # Capa de salida: n_outputs neuronas lineales (6 contaminantes)
    outputs = keras.layers.Dense(
        n_outputs, activation="linear", name="salida_contaminantes"
    )(x)

    modelo = keras.Model(inputs=inputs, outputs=outputs, name="SurrogateAQ_MLP")
    modelo.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
        metrics=["mae"],
    )
    return modelo


# ──────────────────────────────────────────────────────────────────────────────
# 7.  FIGURAS
# ──────────────────────────────────────────────────────────────────────────────

def _estilo_ax(ax: plt.Axes, titulo: str, xlabel: str, ylabel: str) -> None:
    """Aplica tema oscuro consistente a un eje matplotlib."""
    if titulo:
        ax.set_title(titulo, fontsize=10, fontweight="bold", color=TEXT_PLOT, pad=10)
    if xlabel:
        ax.set_xlabel(xlabel, color=TEXT_PLOT, fontsize=9)
    if ylabel:
        ax.set_ylabel(ylabel, color=TEXT_PLOT, fontsize=9)
    ax.tick_params(colors=TEXT_PLOT, labelsize=8)
    ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)


# ── [DL-4]  CURVA DE APRENDIZAJE GLOBAL ÚNICA ────────────────────────────────
def _fig_curva_aprendizaje_dl(
    history: dict,
    parroquia: str = "",
    epochs_real: int = 0,
) -> plt.Figure:
    """
    Una sola figura con dos subgráficas:
        Izquierda : MSE (Train Loss / Val Loss) por época
        Derecha   : MAE (Train / Val)           por época

    La línea continua representa el entrenamiento y la discontinua la validación.
    Se marca la mejor época (mínimo val_loss) con una línea vertical punteada.
    """
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), facecolor=BG_PLOT)

    train_loss = history.get("loss", [])
    val_loss   = history.get("val_loss", [])
    train_mae  = history.get("mae", [])
    val_mae    = history.get("val_mae", [])
    ep         = np.arange(1, len(train_loss) + 1)

    def _trazar_metrica(ax, tr, va, ylabel, etiqueta_tr, etiqueta_va):
        ax.set_facecolor(BG_PLOT)
        if tr:
            ax.plot(ep[:len(tr)], tr, color=COLOR_TRAIN, lw=2.3,
                    label=etiqueta_tr, zorder=5)
        if va:
            ax.plot(ep[:len(va)], va, color=COLOR_VAL, lw=2.3,
                    linestyle="--", label=etiqueta_va, zorder=5)
            best_ep  = int(np.argmin(va)) + 1
            best_val = float(np.min(va))
            ax.axvline(best_ep, color="#F59E0B", lw=1.8, linestyle=":",
                       alpha=0.85, label=f"Mejor época: {best_ep}  ({best_val:.5f})",
                       zorder=6)
            offset = max(1, int(len(ep) * 0.03))
            ax.annotate(
                f" ep.{best_ep}\n {best_val:.4f}",
                xy=(best_ep, best_val),
                xytext=(best_ep + offset, best_val),
                color="#F59E0B", fontsize=8.5, va="center", zorder=7,
            )
        ax.legend(framealpha=0.20, labelcolor=TEXT_PLOT,
                  facecolor=BG_PLOT, edgecolor=GRID_PLOT, fontsize=9)
        _estilo_ax(ax, "", "Época", ylabel)

    _trazar_metrica(
        axes[0], train_loss, val_loss,
        "MSE  (↓ mejor)",
        f"Evolución del Entrenamiento — {parroquia}  (MSE)",
        f"Evolución de la Validación  — {parroquia}  (MSE)",
    )
    _trazar_metrica(
        axes[1], train_mae, val_mae,
        "MAE  (↓ mejor)",
        f"Evolución del Entrenamiento — {parroquia}  (MAE)",
        f"Evolución de la Validación  — {parroquia}  (MAE)",
    )

    nota = (
        f"Red Neuronal MLP · Arquitectura: Input → Dense(256) → Dense(128) → Dense(64) → Output(6) "
        f"· {epochs_real} épocas efectivas (EarlyStopping)"
    )
    fig.text(0.012, 0.012, nota, fontsize=7.5, color="#64748B", ha="left", va="bottom")

    titulo = f"Curva de Aprendizaje Global — MLP Multi-Salida  ·  [{parroquia}]"
    fig.suptitle(titulo, fontsize=12, fontweight="bold", color=TEXT_PLOT, y=1.01)
    plt.tight_layout(rect=[0, 0.04, 1, 1])
    return fig


# ── FIGURA: REAL VS PREDICHO (SCATTER 2×3 para los 6 contaminantes) ──────────
def _fig_prediccion_multi(
    Y_real: np.ndarray,
    Y_pred: np.ndarray,
    target_cols: list[str],
    parroquia: str = "",
) -> plt.Figure:
    """Scatter plot Real vs Predicho para cada contaminante en una cuadrícula 2×3."""
    n     = len(target_cols)
    ncols = 3
    nrows = max(1, (n + ncols - 1) // ncols)

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(14, 5.2 * nrows),
        facecolor=BG_PLOT,
    )
    axes_flat = axes.flatten() if hasattr(axes, "flatten") else [axes]

    for i, cont in enumerate(target_cols):
        ax = axes_flat[i]
        ax.set_facecolor(BG_PLOT)
        yt = Y_real[:, i]
        yp = Y_pred[:, i]

        ax.scatter(yt, yp, alpha=0.25, s=7, color=COLORES_6[i % len(COLORES_6)])
        mn = min(float(yt.min()), float(yp.min()))
        mx = max(float(yt.max()), float(yp.max()))
        ax.plot([mn, mx], [mn, mx], "--", color="#94A3B8", lw=1.5, alpha=0.8,
                label="Ideal (y = ŷ)")

        r2   = r2_score(yt, yp)
        rmse = float(np.sqrt(mean_squared_error(yt, yp)))
        mae  = float(mean_absolute_error(yt, yp))
        ax.set_title(
            f"{cont}   R²={r2:.3f}  RMSE={rmse:.3f}  MAE={mae:.3f}",
            color=TEXT_PLOT, fontsize=9.5, fontweight="bold",
        )
        ax.legend(framealpha=0.15, labelcolor=TEXT_PLOT,
                  facecolor=BG_PLOT, edgecolor=GRID_PLOT, fontsize=8)
        _estilo_ax(ax, "", "Real", "Predicho")

    # Ocultar ejes sobrantes
    for j in range(n, len(axes_flat)):
        axes_flat[j].set_visible(False)

    fig.suptitle(
        f"Real vs Predicho — 6 Contaminantes (Test Set)  ·  [{parroquia}]",
        fontsize=12, fontweight="bold", color=TEXT_PLOT,
    )
    plt.tight_layout()
    return fig


# ── FIGURA: HEATMAP DE CORRELACIONES ─────────────────────────────────────────
def _fig_heatmap(
    df_X: pd.DataFrame,
    df_Y: pd.DataFrame,
    parroquia: str = "",
) -> plt.Figure:
    """Correlación de Pearson entre X e Y (limitada a 25 columnas X más variables)."""
    # Seleccionar las X más variables para no saturar el heatmap
    n_max_x = 20
    if df_X.shape[1] > n_max_x:
        top_x = df_X.std().nlargest(n_max_x).index.tolist()
        df_X_plot = df_X[top_x]
    else:
        df_X_plot = df_X

    df_combined = pd.concat([df_X_plot, df_Y], axis=1)
    corr = df_combined.corr(method="pearson")
    n    = len(corr)

    fig, ax = plt.subplots(
        figsize=(max(10, n * 0.65), max(9, n * 0.60)),
        facecolor=BG_PLOT,
    )
    ax.set_facecolor(BG_PLOT)
    mask = np.zeros_like(corr, dtype=bool)
    mask[np.triu_indices_from(mask, k=1)] = True
    cmap = sns.diverging_palette(230, 20, as_cmap=True)
    sns.heatmap(
        corr, mask=mask, cmap=cmap, vmin=-1, vmax=1, center=0,
        annot=(n <= 22), fmt=".2f",
        annot_kws={"size": 7.0, "color": TEXT_PLOT},
        linewidths=0.35, linecolor=GRID_PLOT,
        square=True, ax=ax,
        cbar_kws={"shrink": 0.60},
    )
    # Resaltar columnas Y con un borde naranja
    y_indices = [list(corr.columns).index(c) for c in df_Y.columns if c in corr.columns]
    for idx in y_indices:
        ax.add_patch(plt.Rectangle(
            (idx, 0), 1, n, fill=False, edgecolor="#F97316", lw=2.2, clip_on=False
        ))
        ax.add_patch(plt.Rectangle(
            (0, idx), n, 1, fill=False, edgecolor="#F97316", lw=2.2, clip_on=False
        ))

    ax.set_title(
        f"Correlación de Pearson — X ↔ Y  ·  [{parroquia}]",
        fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=14,
    )
    ax.tick_params(colors=TEXT_PLOT, labelsize=8)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    plt.setp(ax.get_yticklabels(), rotation=0)
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(colors=TEXT_PLOT, labelsize=8)
    plt.tight_layout()
    return fig


# ──────────────────────────────────────────────────────────────────────────────
# 8.  PIPELINE PRINCIPAL DE ENTRENAMIENTO
# ──────────────────────────────────────────────────────────────────────────────
def entrenar(
    csv_local:        str,
    csv_upload,
    nombre_modelo:    str,
    train_ratio:      float,
    excluir_pandemia: bool,
    epochs_config:    int,
    batch_size:       int,
):
    """
    Pipeline MLOps completo:
        1. Carga CSV  →  detecta timestamp
        2. [DL-1] Separa X/Y dinámicamente
        3. [DL-2] Imputación temporal síncrona
        4. Escala X e Y con StandardScaler (sin data leakage)
        5. [DL-3] Construye y entrena MLP multi-salida
        6. [DL-4] Genera curva de aprendizaje global única
        7. Persiste modelo + metadatos como un único archivo por parroquia
    """
    logs: list[str] = []

    def info(m): log.info(m);    logs.append(f"✅ {m}")
    def warn(m): log.warning(m); logs.append(f"⚠️  {m}")
    def err(m):  log.error(m);   logs.append(f"❌ {m}")
    def _estado(): return "**Registro:**  " + "  ·  ".join(logs[-12:])

    VACIO = (None, None, None)   # fig_pred, fig_hm, fig_lc

    if not TF_DISPONIBLE:
        err("TensorFlow no instalado")
        return (
            "### ❌ TensorFlow no disponible\n"
            "```\npip install tensorflow\n```",
            *VACIO, _estado(),
        )

    try:
        # ── 8.1  Resolver ruta del archivo ────────────────────────────────────
        if csv_upload is not None:
            ruta_csv = csv_upload if isinstance(csv_upload, str) else csv_upload.name
            info(f"Archivo subido: {Path(ruta_csv).name}")
        elif csv_local and not csv_local.startswith("(No"):
            ruta_csv = csv_local
            info(f"Archivo local: {ruta_csv}")
        else:
            err("Sin archivo de entrada")
            return "### ❌ No se seleccionó ningún archivo.", *VACIO, _estado()

        if not Path(ruta_csv).exists():
            err(f"Archivo no encontrado: {ruta_csv}")
            return f"### ❌ Archivo no encontrado: `{ruta_csv}`", *VACIO, _estado()

        nombre_modelo = (nombre_modelo or "surrogate_deep_learning").strip()
        parroquia     = Path(ruta_csv).stem.replace("_", " ").title()

        # ── 8.2  Cargar CSV y preparar índice temporal ───────────────────────
        df = _cargar_csv(ruta_csv)
        info(f"CSV cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")

        ts_col = _detectar_timestamp(df)
        if ts_col is None:
            err("No se detectó columna de tiempo")
            return "### ❌ No se encontró columna Timestamp/Fecha.", *VACIO, _estado()
        info(f"Timestamp detectado: '{ts_col}'")

        df[ts_col] = pd.to_datetime(
            df[ts_col], infer_datetime_format=True, errors="coerce"
        )
        df = df.dropna(subset=[ts_col]).set_index(ts_col).sort_index()

        # ── 8.3  Filtro de pandemia ───────────────────────────────────────────
        if excluir_pandemia:
            mask_pand = df.index.year.isin(ANOS_PANDEMIA)
            n_exc = int(mask_pand.sum())
            df = df[~mask_pand]
            if n_exc:
                info(f"Pandemia excluida: {n_exc:,} registros de {ANOS_PANDEMIA}")

        # ── 8.4  [DL-1]  Detección dinámica de esquema X / Y ─────────────────
        # Convertir columnas objeto a numérico antes de la separación
        obj_cols = df.select_dtypes(include=["object", "string"]).columns
        if len(obj_cols):
            df[obj_cols] = df[obj_cols].apply(pd.to_numeric, errors="coerce")

        x_cols, y_cols = _detectar_columnas_xy(df, ts_col)

        if not y_cols:
            err("No se encontraron columnas Y (PM25, PM10, O3, CO, NO2, SO2)")
            return (
                "### ❌ El dataset no contiene ningún contaminante de la lista "
                f"`{CONTAMINANTES_Y}`.",
                *VACIO, _estado(),
            )
        if not x_cols:
            err("No se encontraron columnas X numéricas")
            return "### ❌ No se detectaron features de entrada numéricas.", *VACIO, _estado()

        info(f"[DL-1] Y ({len(y_cols)} targets): {y_cols}")
        info(f"[DL-1] X ({len(x_cols)} features detectadas)")

        df_X = df[x_cols].copy()
        df_Y = df[y_cols].copy()

        # ── 8.5  [DL-2]  Imputación temporal síncrona ─────────────────────────
        df_X, df_Y = _imputar_temporal(df_X, df_Y)
        info(
            f"[DL-2] Imputación completada — "
            f"NaN residuales: X={int(df_X.isna().sum().sum())} · "
            f"Y={int(df_Y.isna().sum().sum())}"
        )

        # Guardar último registro para inyección de lags en inferencia
        last_row    = df_X.iloc[-1].to_dict()
        ultimo_mes  = df_X.index[-1].month if hasattr(df_X.index, "month") \
                      else datetime.now().month

        # ── 8.6  División temporal Train / Test ──────────────────────────────
        corte = int(len(df_X) * train_ratio)
        X_tr_raw = df_X.iloc[:corte]
        X_te_raw = df_X.iloc[corte:]
        Y_tr_raw = df_Y.iloc[:corte]
        Y_te_raw = df_Y.iloc[corte:]
        info(f"Split temporal: Train={len(X_tr_raw):,} · Test={len(X_te_raw):,}")

        # ── 8.7  Escalado (ajustado solo con train — sin data leakage) ────────
        scaler_x = StandardScaler()
        scaler_y = StandardScaler()
        X_tr = scaler_x.fit_transform(X_tr_raw.values)
        X_te = scaler_x.transform(X_te_raw.values)
        Y_tr = scaler_y.fit_transform(Y_tr_raw.values)
        Y_te = scaler_y.transform(Y_te_raw.values)

        # ── 8.8  [DL-3]  Construir y entrenar la Red Neuronal ─────────────────
        modelo = _construir_red_neuronal(X_tr.shape[1], len(y_cols))
        info(
            f"[DL-3] MLP construida: "
            f"Input({X_tr.shape[1]}) → [256→128→64] → Output({len(y_cols)})"
        )
        # Resumen de parámetros
        total_params = modelo.count_params()
        info(f"Parámetros totales: {total_params:,}")

        callbacks_list = [
            keras.callbacks.EarlyStopping(
                patience=25,
                restore_best_weights=True,
                monitor="val_loss",
                verbose=0,
            ),
            keras.callbacks.ReduceLROnPlateau(
                patience=12,
                factor=0.5,
                monitor="val_loss",
                verbose=0,
                min_lr=1e-7,
            ),
        ]

        info(f"Entrenando: epochs_max={epochs_config} · batch={batch_size}…")
        history = modelo.fit(
            X_tr, Y_tr,
            validation_split=0.15,
            epochs=epochs_config,
            batch_size=batch_size,
            callbacks=callbacks_list,
            verbose=0,
        )
        epochs_real = len(history.history["loss"])
        final_loss  = history.history["val_loss"][-1]
        info(
            f"[DL-3] Entrenamiento finalizado: {epochs_real} épocas efectivas  "
            f"· Val MSE final = {final_loss:.6f}"
        )

        # ── 8.9  Evaluación por contaminante ──────────────────────────────────
        Y_pred_te_sc = modelo.predict(X_te, verbose=0)
        Y_pred_tr_sc = modelo.predict(X_tr, verbose=0)
        Y_pred_te    = scaler_y.inverse_transform(Y_pred_te_sc)
        Y_pred_tr    = scaler_y.inverse_transform(Y_pred_tr_sc)
        Y_te_orig    = scaler_y.inverse_transform(Y_te)
        Y_tr_orig    = scaler_y.inverse_transform(Y_tr)

        filas_met: list[dict] = []
        for i, cont in enumerate(y_cols):
            m_tr = {
                "MAE":  mean_absolute_error(Y_tr_orig[:, i], Y_pred_tr[:, i]),
                "RMSE": float(np.sqrt(mean_squared_error(Y_tr_orig[:, i], Y_pred_tr[:, i]))),
                "R2":   r2_score(Y_tr_orig[:, i], Y_pred_tr[:, i]),
            }
            m_te = {
                "MAE":  mean_absolute_error(Y_te_orig[:, i], Y_pred_te[:, i]),
                "RMSE": float(np.sqrt(mean_squared_error(Y_te_orig[:, i], Y_pred_te[:, i]))),
                "R2":   r2_score(Y_te_orig[:, i], Y_pred_te[:, i]),
            }
            filas_met.append({"Contaminante": cont, **m_tr, **m_te,
                               "delta_R2": m_tr["R2"] - m_te["R2"]})

        r2_global_te = float(np.mean([r["R2"] for r in filas_met]))
        r2_global_tr = float(np.mean([m["R2"] for m in
                              [{"R2": r2_score(Y_tr_orig[:, i], Y_pred_tr[:, i])}
                               for i in range(len(y_cols))]]))

        # ── 8.10  Markdown de métricas ────────────────────────────────────────
        filas_tabla = []
        for r in filas_met:
            r2_icon  = "🟢" if r["R2"] >= 0.7 else ("🟡" if r["R2"] >= 0.5 else "🔴")
            gap_icon = "⚠️" if r["delta_R2"] > 0.15 else "✅"
            # Note: dict keys from the merge are Train MAE/RMSE/R2 and Test MAE/RMSE/R2
            # We stored m_tr keys directly → need to re-compute
            idx = y_cols.index(r["Contaminante"])
            tr_mae  = mean_absolute_error(Y_tr_orig[:, idx], Y_pred_tr[:, idx])
            tr_rmse = float(np.sqrt(mean_squared_error(Y_tr_orig[:, idx], Y_pred_tr[:, idx])))
            tr_r2   = r2_score(Y_tr_orig[:, idx], Y_pred_tr[:, idx])
            te_mae  = mean_absolute_error(Y_te_orig[:, idx], Y_pred_te[:, idx])
            te_rmse = float(np.sqrt(mean_squared_error(Y_te_orig[:, idx], Y_pred_te[:, idx])))
            te_r2   = r2_score(Y_te_orig[:, idx], Y_pred_te[:, idx])
            gap     = tr_r2 - te_r2
            filas_tabla.append(
                f"| **{r['Contaminante']}** "
                f"| `{tr_mae:.3f}` | `{tr_rmse:.3f}` | `{tr_r2:.3f}` "
                f"| `{te_mae:.3f}` | `{te_rmse:.3f}` | {r2_icon} `{te_r2:.3f}` "
                f"| {gap_icon} `{gap:.3f}` |"
            )

        tabla_md = "\n".join(filas_tabla)
        hw_label = f"{'🟢 GPU' if GPU_DISPONIBLE else '🔵 CPU'} — {GPU_MSG}"
        pand_lbl = "🚫 2020-2021 excluidos" if excluir_pandemia else "⚠️ Pandemia incluida"

        metricas_md = f"""
## 🧠 Deep Learning Surrogate — *{parroquia}*

### Red Neuronal MLP — {len(y_cols)} Contaminantes Simultáneos

| Contaminante | Train MAE | Train RMSE | Train R² | Test MAE | Test RMSE | Test R² | ΔR² |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
{tabla_md}

> **Arquitectura:** `Input({len(x_cols)}) → Dense(256,BN,ReLU,Dr.30%) → Dense(128,BN,ReLU,Dr.20%) → Dense(64,BN,ReLU,Dr.10%) → Output({len(y_cols)})`
> **EarlyStopping** (patience=25): `{epochs_real}` épocas efectivas de `{epochs_config}` configuradas
> **R² Global Test:** `{r2_global_te:.4f}`  ·  **Val MSE final:** `{final_loss:.6f}`

---

| Parámetro | Valor |
|-----------|-------|
| Parroquia | `{parroquia}` |
| Hardware  | {hw_label} |
| Parámetros del modelo | `{total_params:,}` |
| Features X detectadas | `{len(x_cols)}` |
| Contaminantes Y | `{y_cols}` |
| Train / Test | `{len(X_tr):,}` / `{len(X_te):,}` registros |
| Épocas efectivas | `{epochs_real}` |
| Batch size | `{batch_size}` |
| Filtro pandemia | {pand_lbl} |
"""

        # ── 8.11  [DL-3]  Persistencia: un único archivo por parroquia ────────
        tag        = f"{nombre_modelo}_{parroquia.replace(' ', '_')}"
        model_path = OUTPUT_DIR / f"{tag}_deep_learning_model.keras"
        meta_path  = OUTPUT_DIR / f"{tag}_meta.pkl"

        modelo.save(str(model_path))

        with open(meta_path, "wb") as fh:
            pickle.dump(
                {
                    "scaler_x":   scaler_x,
                    "scaler_y":   scaler_y,
                    "feat_cols":  x_cols,
                    "target_cols": y_cols,
                    "parroquia":  parroquia,
                    "last_row":   last_row,
                    "ultimo_mes": ultimo_mes,
                    "metricas":   filas_met,
                    "epochs_real": epochs_real,
                },
                fh,
            )
        info(f"Modelo guardado → {model_path.name}")
        info(f"Metadatos guardados → {meta_path.name}")

        # ── 8.12  Actualizar estado de sesión ─────────────────────────────────
        SESION.update(
            {
                "modelo":      modelo,
                "scaler_x":    scaler_x,
                "scaler_y":    scaler_y,
                "feat_cols":   x_cols,
                "target_cols": y_cols,
                "parroquia":   parroquia,
                "last_row":    last_row,
                "ultimo_mes":  ultimo_mes,
                "ruta_modelo": str(model_path),
            }
        )
        info("Modelo en sesión → pestaña Predicción habilitada.")

        # ── 8.13  Figuras ─────────────────────────────────────────────────────
        fig_lc   = _fig_curva_aprendizaje_dl(history.history, parroquia, epochs_real)
        fig_pred = _fig_prediccion_multi(Y_te_orig, Y_pred_te, y_cols, parroquia)
        fig_hm   = _fig_heatmap(df_X, df_Y, parroquia)

        return metricas_md, fig_pred, fig_hm, fig_lc, _estado()

    except ImportError as e:
        err(str(e))
        pkg = str(e).split("'")[-2] if "'" in str(e) else str(e).split()[-1]
        return (
            f"### ❌ Librería faltante\n```\n{e}\n```\n"
            f"Instala con: `pip install {pkg}`",
            *VACIO, _estado(),
        )
    except Exception as e:
        err(str(e))
        return (
            f"### ❌ Error inesperado\n```\n{traceback.format_exc()}\n```",
            *VACIO, _estado(),
        )


# ──────────────────────────────────────────────────────────────────────────────
# 9.  [DL-5]  INFERENCIA LIMPIA — 6 CONTAMINANTES SIMULTÁNEOS
# ──────────────────────────────────────────────────────────────────────────────

def _icono_calidad(cont: str, valor: float) -> str:
    """Devuelve un emoji de calidad del aire según umbrales EPA."""
    umbrales = {
        "PM25": [(12, "🟢"), (35, "🟡"), (55, "🟠"), (150, "🔴"), (float("inf"), "🟣")],
        "PM10": [(54, "🟢"), (154, "🟡"), (254, "🟠"), (354, "🔴"), (float("inf"), "🟣")],
        "O3":   [(54, "🟢"), (70,  "🟡"), (85,  "🟠"), (105,  "🔴"), (float("inf"), "🟣")],
        "NO2":  [(53, "🟢"), (100, "🟡"), (360, "🟠"), (649,  "🔴"), (float("inf"), "🟣")],
        "CO":   [(4.4, "🟢"), (9.4, "🟡"), (12.4, "🟠"), (15.4, "🔴"), (float("inf"), "🟣")],
        "SO2":  [(35, "🟢"), (75,  "🟡"), (185, "🟠"), (304,  "🔴"), (float("inf"), "🟣")],
    }
    for umbral, icono in umbrales.get(cont, [(float("inf"), "⚪")]):
        if valor <= umbral:
            return icono
    return "⚪"


def predecir_contaminantes(
    temperatura: float,
    humedad: float,
    viento_vel: float,
    viento_dir: float,
    precipitacion: float,
    hora: int,
) -> str:
    """
    [DL-5]  Recibe solo las variables climáticas básicas y la hora.

    Backend automático:
        1. Calcula variables cíclicas (hora_sin/cos, mes_sin/cos).
        2. Inyecta los últimos lags históricos válidos almacenados en SESION.
        3. Construye el vector completo alineado con el esquema de entrenamiento.
        4. Escala, predice y devierte la escala.

    Retorna una tabla Markdown con los 6 contaminantes calculados simultáneamente.
    """
    if SESION["modelo"] is None:
        return (
            "### ⚠️ Sin modelo en sesión\n"
            "Entrena primero un modelo en la pestaña **🚀 Entrenamiento**."
        )

    try:
        feat_cols   = SESION["feat_cols"]
        scaler_x    = SESION["scaler_x"]
        scaler_y    = SESION["scaler_y"]
        target_cols = SESION["target_cols"]
        parroquia   = SESION["parroquia"]
        last_row    = SESION["last_row"]
        mes         = SESION["ultimo_mes"]

        # ── Calcular variables cíclicas ───────────────────────────────────────
        hora_sin = float(np.sin(2 * np.pi * hora / 24))
        hora_cos = float(np.cos(2 * np.pi * hora / 24))
        mes_sin  = float(np.sin(2 * np.pi * mes / 12))
        mes_cos  = float(np.cos(2 * np.pi * mes / 12))

        # Mapa de variables provistas por el usuario
        vars_usuario = {
            "Temperatura":      float(temperatura),
            "Humedad":          float(humedad),
            "Viento_Velocidad": float(viento_vel),
            "Viento_Direccion": float(viento_dir),
            "Precipitacion":    float(precipitacion),
            "hora_sin":  hora_sin,
            "hora_cos":  hora_cos,
            "mes_sin":   mes_sin,
            "mes_cos":   mes_cos,
        }

        # ── Construir vector completo (injección de lags) ─────────────────────
        row: dict = {}
        for col in feat_cols:
            if col in vars_usuario:
                row[col] = vars_usuario[col]
            elif col in last_row:
                # Inyectar último valor conocido (lags históricos y otros)
                row[col] = float(last_row[col])
            else:
                row[col] = 0.0   # fallback seguro

        X_input  = pd.DataFrame([row])[feat_cols]
        X_scaled = scaler_x.transform(X_input.values)

        Y_scaled = SESION["modelo"].predict(X_scaled, verbose=0)
        Y_pred   = scaler_y.inverse_transform(Y_scaled)[0]

        # ── Construir tabla de salida ─────────────────────────────────────────
        unidades = {
            "PM25": "µg/m³", "PM10": "µg/m³", "O3": "µg/m³",
            "CO": "mg/m³", "NO2": "µg/m³", "SO2": "µg/m³",
        }
        filas_tab = []
        for i, cont in enumerate(target_cols):
            val  = float(Y_pred[i])
            icon = _icono_calidad(cont, val)
            unit = unidades.get(cont, "µg/m³")
            filas_tab.append(
                f"| {icon} **{cont}** | `{val:.3f}` | `{unit}` |"
            )
        tabla = "\n".join(filas_tab)

        return f"""
## 🔮 Predicción Multi-Contaminante — *{parroquia}*

| Contaminante | Concentración Estimada | Unidad |
|:---:|:---:|:---:|
{tabla}

---

**Condiciones ingresadas:**

| Variable | Valor |
|----------|-------|
| Temperatura | `{temperatura} °C` |
| Humedad Relativa | `{humedad} %` |
| Velocidad Viento | `{viento_vel} m/s` |
| Dirección Viento | `{viento_dir} °` |
| Precipitación | `{precipitacion} mm` |
| Hora | `{hora:02d}:00` |
| Mes (último conocido) | `{mes}` |

*Los lags históricos fueron inyectados automáticamente desde el último registro del dataset.*
*Modelo: `{SESION['ruta_modelo'].split('/')[-1]}`*
"""
    except Exception:
        return f"### ❌ Error durante la predicción\n```\n{traceback.format_exc()}\n```"


# ──────────────────────────────────────────────────────────────────────────────
# 10.  CSS
# ──────────────────────────────────────────────────────────────────────────────
CSS = """
:root {
    --bg:      #0F172A;
    --card:    #1E293B;
    --input:   #0D1525;
    --border:  #334155;
    --accent:  #6366F1;
    --accentH: #818CF8;
    --text:    #F1F5F9;
    --muted:   #94A3B8;
    --ok:      #22C55E;
    --warn:    #F59E0B;
    --err:     #EF4444;
    --r:       10px;
}
body, .gradio-container { background:var(--bg) !important; color:var(--text) !important;
    font-family:'Inter','Segoe UI',sans-serif !important; }
.gr-group, .gr-box { background:var(--card) !important;
    border:1px solid var(--border) !important; border-radius:var(--r) !important;
    padding:16px !important; }
input, textarea, select { background:var(--input) !important; color:var(--text) !important;
    border:1px solid var(--border) !important; border-radius:6px !important; }
label, .gr-label { color:var(--muted) !important; font-size:.8rem !important;
    font-weight:700 !important; text-transform:uppercase; letter-spacing:.05em !important; }
button.primary { background:linear-gradient(135deg,var(--accent),#7C3AED) !important;
    color:#fff !important; border:none !important; border-radius:8px !important;
    font-weight:800 !important; font-size:1rem !important; padding:12px 28px !important;
    box-shadow:0 4px 20px rgba(99,102,241,.45); transition:all .15s; }
button.primary:hover { transform:translateY(-2px); box-shadow:0 6px 28px rgba(99,102,241,.6); }
button.secondary { background:var(--card) !important; color:var(--text) !important;
    border:1px solid var(--border) !important; border-radius:8px !important; }
.gr-markdown { color:var(--text) !important; }
.gr-markdown table { border-collapse:collapse; width:100%; }
.gr-markdown th { background:#1E293B; color:var(--accentH); padding:8px 14px;
    border:1px solid var(--border); }
.gr-markdown td { color:var(--text); padding:7px 14px; border:1px solid var(--border); }
.gr-markdown tr:nth-child(even) td { background:#19253a; }
.gr-file { border:2px dashed var(--accent) !important; border-radius:var(--r) !important; }
.hero { text-align:center; padding:24px 0 6px; }
.hero h1 { font-size:2rem; font-weight:900;
    background:linear-gradient(90deg,#6366F1,#38BDF8);
    -webkit-background-clip:text; -webkit-text-fill-color:transparent; }
.hero p { color:var(--muted); font-size:.88rem; }
.badge { display:inline-block; padding:4px 12px; border-radius:20px; font-size:.75rem;
    font-weight:700; margin-top:4px; }
.badge-gpu  { background:rgba(34,197,94,.18);  color:#4ADE80; border:1px solid #22C55E; }
.badge-cpu  { background:rgba(99,102,241,.15); color:#A5B4FC; border:1px solid #6366F1; }
.badge-tf   { background:rgba(249,115,22,.15); color:#FB923C; border:1px solid #F97316; }
.logs-box   { background:#0B1527 !important; border:1px solid #1D3557 !important;
    border-radius:8px; padding:10px 14px; font-family:'JetBrains Mono',monospace;
    font-size:.76rem; color:#7DD3FC; max-height:130px; overflow-y:auto; line-height:1.6; }
"""


# ──────────────────────────────────────────────────────────────────────────────
# 11.  CONSTRUCCIÓN DE LA UI GRADIO
# ──────────────────────────────────────────────────────────────────────────────

gpu_badge_html = (
    f'<span class="badge badge-gpu">🟢 GPU: {GPU_MSG.split(": ")[-1]}</span>'
    if GPU_DISPONIBLE else
    f'<span class="badge badge-cpu">🔵 CPU — {GPU_MSG}</span>'
)
tf_badge_html = (
    '<span class="badge badge-tf">🔶 TensorFlow disponible</span>'
    if TF_DISPONIBLE else
    '<span class="badge badge-cpu" style="border-color:#EF4444;color:#F87171">⚠️ TF no instalado</span>'
)


def construir_app() -> gr.Blocks:
    with gr.Blocks(
        theme=gr.themes.Base(
            primary_hue="indigo",
            secondary_hue="sky",
            neutral_hue="slate",
            font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif"],
        ),
        css=CSS,
        title="Surrogate DL — Calidad del Aire v6",
    ) as app:

        # ── Hero ──────────────────────────────────────────────────────────────
        gr.HTML(f"""
        <div class="hero">
            <h1>🌬️ Surrogate Model — Calidad del Aire</h1>
            <p>Red Neuronal MLP Multi-Salida · 6 Contaminantes Simultáneos · v6.0 — Deep Learning</p>
            {gpu_badge_html} &nbsp; {tf_badge_html}
        </div>
        """)

        with gr.Row(equal_height=False):

            # ═══════════════════════════════════
            # PANEL IZQUIERDO — CONTROLES
            # ═══════════════════════════════════
            with gr.Column(scale=1, min_width=360):

                # ── Dataset ──────────────────────────────────────────────────
                with gr.Group():
                    gr.Markdown("### 📂 Dataset")
                    with gr.Row():
                        csv_dropdown = gr.Dropdown(
                            label="CSVs detectados en el servidor",
                            choices=listar_csvs(),
                            value=None,
                            interactive=True,
                            scale=5,
                        )
                        btn_refresh = gr.Button(
                            "🔄", scale=1, min_width=46, variant="secondary"
                        )
                    csv_upload = gr.File(
                        label="O arrastra / sube un CSV externo",
                        file_types=[".csv"],
                        type="filepath",
                    )
                    btn_detectar = gr.Button(
                        "🔍 Inspeccionar esquema X/Y del CSV",
                        variant="secondary",
                        size="sm",
                    )
                    info_esquema = gr.Markdown(
                        "_Pulsa **Inspeccionar** para ver la separación X/Y detectada._"
                    )

                gr.HTML("<div style='height:8px'/>")

                # ── Configuración del Modelo ──────────────────────────────────
                with gr.Group():
                    gr.Markdown("### 🧠 Configuración de la Red Neuronal")
                    gr.Markdown(
                        "> **Arquitectura fija:** "
                        "`Input → Dense(256,BN) → Dense(128,BN) → Dense(64,BN) → Output(6)`"
                    )
                    nombre_modelo_input = gr.Textbox(
                        label="Nombre del modelo (archivo .keras)",
                        value="surrogate_calidad_aire",
                    )
                    with gr.Row():
                        epochs_slider = gr.Slider(
                            label="Épocas máximas",
                            minimum=50,
                            maximum=1000,
                            step=50,
                            value=300,
                            scale=3,
                        )
                        batch_slider = gr.Slider(
                            label="Batch Size",
                            minimum=32,
                            maximum=512,
                            step=32,
                            value=128,
                            scale=2,
                        )
                    with gr.Row():
                        train_ratio_slider = gr.Slider(
                            label="Proporción de entrenamiento",
                            minimum=0.6,
                            maximum=0.95,
                            step=0.05,
                            value=0.80,
                            scale=3,
                        )
                        excluir_pandemia_chk = gr.Checkbox(
                            label="🚫 Excluir pandemia (2020-2021)",
                            value=True,
                            scale=2,
                        )

                gr.HTML("<div style='height:10px'/>")
                btn_train = gr.Button(
                    "🚀  Entrenar Red Neuronal",
                    variant="primary",
                    size="lg",
                )
                gr.HTML("<div style='height:6px'/>")
                gr.Markdown("##### 🖥️ Registro de ejecución")
                estado_output = gr.Markdown(
                    value="_Esperando ejecución…_",
                    elem_classes=["logs-box"],
                )

            # ═══════════════════════════════════
            # PANEL DERECHO — RESULTADOS (5 tabs)
            # ═══════════════════════════════════
            with gr.Column(scale=2, min_width=600):
                with gr.Tabs():

                    # Tab 1: Métricas ─────────────────────────────────────────
                    with gr.Tab("📊 Métricas"):
                        metricas_output = gr.Markdown(
                            value="*Las métricas aparecerán aquí tras entrenar.*"
                        )

                    # Tab 2: Real vs Predicho ─────────────────────────────────
                    with gr.Tab("📈 Real vs Predicho"):
                        gr.Markdown(
                            "**Scatter plots** Real vs Predicho para los 6 contaminantes "
                            "en el conjunto de test. Un R² cercano a 1 indica buena capacidad "
                            "predictiva. La línea punteada representa la predicción perfecta."
                        )
                        fig_pred_output = gr.Plot(
                            label="Scatter Real vs Predicho — 6 Contaminantes"
                        )

                    # Tab 3: Curva de Aprendizaje [DL-4] ──────────────────────
                    with gr.Tab("📉 Curva de Aprendizaje"):
                        gr.Markdown(
                            "**Progreso del entrenamiento global** de la Red Neuronal MLP. "
                            "La línea **continua** es el error de entrenamiento y la "
                            "línea **discontinua** es el error de validación interna (15 %).\n\n"
                            "La línea vertical punteada indica la **mejor época** "
                            "según `val_loss` (restaurada por EarlyStopping).\n\n"
                            "- **Izquierda:** Función de pérdida MSE (↓ mejor)\n"
                            "- **Derecha:** Error Absoluto Medio MAE (↓ mejor)"
                        )
                        fig_lc_output = gr.Plot(
                            label="Curva de Aprendizaje — Train Loss vs Val Loss (por época)"
                        )

                    # Tab 4: Análisis de Dependencias ─────────────────────────
                    with gr.Tab("📊 Análisis de Dependencias"):
                        gr.Markdown(
                            "**Correlación de Pearson** entre features X y contaminantes Y. "
                            "Las filas/columnas de los contaminantes están resaltadas en naranja. "
                            "Se muestran las 20 variables X de mayor varianza."
                        )
                        fig_hm_output = gr.Plot(
                            label="Mapa de Calor — Correlación de Pearson X ↔ Y"
                        )

                    # Tab 5: Predicción Multi-Salida [DL-5] ───────────────────
                    with gr.Tab("🔮 Predicción Multi-Salida"):
                        gr.Markdown(
                            "Ingresa las **condiciones climáticas actuales** y la hora. "
                            "El backend calculará automáticamente las variables cíclicas e "
                            "inyectará los últimos lags históricos del dataset.\n\n"
                            "La Red Neuronal devolverá los **6 contaminantes** calculados "
                            "simultáneamente en una sola inferencia."
                        )

                        sesion_info = gr.Markdown(
                            "_⚠️ Entrena un modelo para habilitar esta sección._"
                        )

                        with gr.Group():
                            gr.Markdown("##### Variables Climáticas de Entrada")
                            with gr.Row():
                                inp_temp   = gr.Slider(
                                    label="Temperatura (°C)",
                                    minimum=-10, maximum=50, step=0.5, value=20,
                                )
                                inp_hum    = gr.Slider(
                                    label="Humedad Relativa (%)",
                                    minimum=0, maximum=100, step=1, value=60,
                                )
                            with gr.Row():
                                inp_vvel   = gr.Slider(
                                    label="Velocidad del Viento (m/s)",
                                    minimum=0, maximum=30, step=0.5, value=3,
                                )
                                inp_vdir   = gr.Slider(
                                    label="Dirección del Viento (°)",
                                    minimum=0, maximum=360, step=5, value=180,
                                )
                            with gr.Row():
                                inp_precip = gr.Slider(
                                    label="Precipitación (mm)",
                                    minimum=0, maximum=100, step=0.5, value=0,
                                )
                                inp_hora   = gr.Slider(
                                    label="Hora del día (0 – 23)",
                                    minimum=0, maximum=23, step=1, value=12,
                                )

                        btn_predecir  = gr.Button(
                            "🔮  Predecir 6 Contaminantes",
                            variant="primary",
                            size="lg",
                        )
                        resultado_pred = gr.Markdown(
                            value="_El resultado aparecerá aquí tras predecir._"
                        )

        gr.HTML("""
        <div style="text-align:center;padding:18px 0 8px;color:#475569;font-size:.76rem;">
            Surrogate Model v6.0  ·  Deep Learning MLP Multi-Salida  ·  TensorFlow/Keras
            ·  Auto-GPU CUDA/V100  ·  6 Contaminantes REMMAQ Simultáneos
        </div>
        """)

        # ── Función para actualizar sesion_info tras entrenar ─────────────────
        def _actualizar_sesion_info():
            if SESION["modelo"] is None:
                return "_⚠️ Entrena un modelo para habilitar esta sección._"
            y = SESION["target_cols"]
            p = SESION["parroquia"]
            n = len(SESION["feat_cols"])
            r = SESION["ruta_modelo"].split("/")[-1]
            return (
                f"✅ **Modelo en sesión:** *{p}* · "
                f"Targets: `{y}` · "
                f"Features: `{n}` · "
                f"Archivo: `{r}`"
            )

        # ── Eventos ───────────────────────────────────────────────────────────
        btn_refresh.click(
            fn=refrescar_dropdown,
            inputs=[],
            outputs=[csv_dropdown],
        )

        def _wrapper_detectar(csv_loc, csv_up):
            ruta = (
                (csv_up if isinstance(csv_up, str) else csv_up.name)
                if csv_up
                else (csv_loc if csv_loc and not csv_loc.startswith("(No") else None)
            )
            if not ruta:
                return "⚠️ Selecciona o sube un archivo CSV primero."
            return obtener_info_esquema(ruta)

        btn_detectar.click(
            fn=_wrapper_detectar,
            inputs=[csv_dropdown, csv_upload],
            outputs=[info_esquema],
        )

        # ── Entrenamiento → 4 salidas + estado + refresh sesion_info ─────────
        btn_train.click(
            fn=entrenar,
            inputs=[
                csv_dropdown,
                csv_upload,
                nombre_modelo_input,
                train_ratio_slider,
                excluir_pandemia_chk,
                epochs_slider,
                batch_slider,
            ],
            outputs=[
                metricas_output,
                fig_pred_output,
                fig_hm_output,
                fig_lc_output,
                estado_output,
            ],
        ).then(
            fn=_actualizar_sesion_info,
            inputs=[],
            outputs=[sesion_info],
        )

        # ── Predicción ────────────────────────────────────────────────────────
        btn_predecir.click(
            fn=predecir_contaminantes,
            inputs=[
                inp_temp, inp_hum, inp_vvel,
                inp_vdir, inp_precip, inp_hora,
            ],
            outputs=[resultado_pred],
        )

    return app


# ──────────────────────────────────────────────────────────────────────────────
# 12.  PUNTO DE ENTRADA
# ──────────────────────────────────────────────────────────────────────────────

def _ip_fallback() -> None:
    import socket
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        print(f"  🖥️  IP local   : http://{s.getsockname()[0]}:<PUERTO>")
        s.close()
    except Exception:
        pass
    try:
        ips = subprocess.check_output(["hostname", "-I"], text=True, timeout=4).strip()
        print(f"  🌐  IPs servidor: {ips}")
    except Exception:
        pass


if __name__ == "__main__":
    app = construir_app()

    print("\n" + "═" * 66)
    print("  🌬️  SURROGATE MODEL — Calidad del Aire  v6.0")
    print("  🧠  Deep Learning MLP Multi-Salida — 6 Contaminantes Simultáneos")
    print("═" * 66)
    print(f"  TensorFlow  : {'✅ disponible' if TF_DISPONIBLE else '❌ no instalado'}")
    print(f"  Hardware    : {GPU_MSG}")
    print(f"  GPU activa  : {GPU_DISPONIBLE}")
    print("  Puerto      : automático (server_port=None)")
    print("═" * 66)
    print("\n⏳ Generando enlace de acceso…\n")

    try:
        app.launch(
            server_name="0.0.0.0",
            server_port=None,
            share=True,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )
    except OSError as port_err:
        print(f"\n⚠️  Error de puerto: {port_err}")
        print("🔄  Reintentando en puerto 7861…\n")
        app.launch(
            server_name="0.0.0.0",
            server_port=7861,
            share=True,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )
    except Exception as tunnel_err:
        print(f"\n⚠️  Túnel público falló: {tunnel_err}")
        print("🔄  Relanzando sin túnel (acceso por IP directa):\n")
        _ip_fallback()
        app.launch(
            server_name="0.0.0.0",
            server_port=None,
            share=False,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )

2026-06-07 16:57:13.565923: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
16:57:17 [INFO] TensorFlow GPU: 1 dispositivo(s) activo(s) [memory_growth=True]
16:57:17 [INFO] GPU detectada (nvidia-smi): Tesla V100-PCIE-16GB, 16384 MiB


✅ MLOPS: Crecimiento dinámico de memoria inyectado con éxito en la V100.


16:58:44 [INFO] Archivo subido: dataset_ml_carapungo.csv
16:58:45 [INFO] CSV cargado: 160,262 filas × 19 columnas
16:58:45 [INFO] Timestamp detectado: 'Timestamp'
16:58:45 [INFO] Pandemia excluida: 23 registros de [2020, 2021]
16:58:45 [INFO] [DL-1] Y (6 targets): ['PM25', 'PM10', 'O3', 'CO', 'NO2', 'SO2']
16:58:45 [INFO] [DL-1] X (12 features detectadas)
16:58:45 [INFO] [DL-2] Imputación completada — NaN residuales: X=0 · Y=0
16:58:45 [INFO] Split temporal: Train=128,191 · Test=32,048
I0000 00:00:1780851525.537463 2986761 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14791 MB memory:  -> device: 0, name: Tesla V100-PCIE-16GB, pci bus id: 0000:83:00.0, compute capability: 7.0
2026-06-07 16:58:46.091825: W external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:237] Falling back to the CUDA driver for PTX compilation; ptxas does not support CC 7.0
2026-06-07 16:58:46.091842: W external/local_xla/xla/stream_executor/cuda/subprocess_com

Version 1 de deepseek

In [1]:
"""
================================================================================
  SURROGATE MODEL — CALIDAD DEL AIRE  |  Interfaz Web Gradio
  Autor   : Arquitecto de Datos / MLOps Engineer
  Versión : 6.0  (Deep Learning — MLP Tabular Multi-Salida)
  ─── Re-arquitectura v6.0 ─────────────────────────────────────────────────
  [DL-1]  Detección dinámica de esquema X/Y por inspección de cabeceras CSV
  [DL-2]  Imputación temporal síncrona ffill() → bfill() (anti-NaN para NN)
  [DL-3]  MLP Multi-Salida con PyTorch (CUDA V100): 6 neuronas de salida
          Capas: Dense + BatchNorm + ReLU + Dropout — activación lineal final
  [DL-4]  UNA sola curva de aprendizaje global: Train Loss / Val Loss por época
  [DL-5]  Inferencia limpia: solo variables climáticas + hora del usuario
          Backend: auto-calcula cíclicas + inyecta lags históricos del CSV
  ─── Correcciones heredadas de v4.0 ────────────────────────────────────────
  [FIX-1]  CSV con cabeceras decorativas  → comment='#' en read_csv
  [FIX-2]  Inestabilidad en Jupyter/V100  → share=True + blocking correcto
  [FIX-3]  Filtro de pandemia 2020-2021   → exclusión cronológica explícita
  [FIX-4]  Auto-detección GPU             → torch.cuda / nvidia-smi
================================================================================
  Dependencias mínimas:
      pip install gradio scikit-learn pandas numpy matplotlib seaborn
  Dependencias DL (OBLIGATORIAS para v6):
      pip install torch                              # CPU
      # GPU CUDA 11.8 (Tesla V100):
      pip install torch --index-url https://download.pytorch.org/whl/cu118
================================================================================
"""

# ──────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTACIONES
# ──────────────────────────────────────────────────────────────────────────────
import os
import json
import warnings
import logging
import traceback
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import gradio as gr
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# PyTorch — requerido para entrenamiento DL
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_OK = True
except ImportError:
    TORCH_OK = False

warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# ──────────────────────────────────────────────────────────────────────────────
# 1.  CONSTANTES Y CONFIGURACIÓN GLOBAL
# ──────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("resultados_surrogate")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ANOS_PANDEMIA = [2020, 2021]

# ── [DL-1]  Separación académica X / Y ──────────────────────────────────────
# Regla de Oro: ÚNICAMENTE estas columnas pueden pertenecer a Y.
CONTAMINANTES_Y: list[str] = ["PM25", "PM10", "O3", "CO", "NO2", "SO2"]

# Columnas de control / metadatos — se excluyen de X e Y
VARS_CONTROL: set[str] = {
    "Fecha", "fecha", "FECHA",
    "Parroquia", "parroquia", "PARROQUIA",
    "Estacion", "estacion", "ESTACION",
    "id", "ID", "Id",
}

# Variables climáticas que el usuario introduce en la inferencia
VARS_CLIMA_INPUT: list[str] = [
    "Temperatura", "Humedad", "Viento_Velocidad",
    "Viento_Direccion", "Precipitacion",
]

# ── Arquitecturas MLP disponibles ────────────────────────────────────────────
ARQUITECTURAS: dict = {
    "Ligera  [256·128·64]": {
        "hidden_dims":   [256, 128, 64],
        "dropout_rates": [0.25, 0.20, 0.15],
    },
    "Estándar  [512·256·128·64]": {
        "hidden_dims":   [512, 256, 128, 64],
        "dropout_rates": [0.30, 0.25, 0.20, 0.15],
    },
    "Profunda  [1024·512·256·128·64]": {
        "hidden_dims":   [1024, 512, 256, 128, 64],
        "dropout_rates": [0.35, 0.30, 0.25, 0.20, 0.15],
    },
}

# ── Umbrales AQI por contaminante (µg/m³ o ppm) ──────────────────────────────
AQI_UMBRALES: dict = {
    "PM25": [(12, "🟢 Buena"), (35, "🟡 Moderada"), (55, "🟠 Insalubre GS"),
             (150, "🔴 Insalubre"), (float("inf"), "🟣 Muy insalubre")],
    "PM10": [(54, "🟢 Buena"), (154, "🟡 Moderada"), (254, "🟠 Insalubre GS"),
             (float("inf"), "🔴 Insalubre")],
    "O3":   [(54, "🟢 Buena"), (70, "🟡 Moderada"), (85, "🟠 Insalubre GS"),
             (float("inf"), "🔴 Insalubre")],
    "CO":   [(9.4, "🟢 Buena"), (12.4, "🟡 Moderada"), (float("inf"), "🔴 Insalubre")],
    "NO2":  [(53, "🟢 Buena"), (100, "🟡 Moderada"), (360, "🟠 Insalubre GS"),
             (float("inf"), "🔴 Insalubre")],
    "SO2":  [(35, "🟢 Buena"), (75, "🟡 Moderada"), (185, "🟠 Insalubre GS"),
             (float("inf"), "🔴 Insalubre")],
}

# ── Colores del tema oscuro ───────────────────────────────────────────────────
COLOR_TRAIN = "#3B82F6"   # azul   — Train Loss
COLOR_VAL   = "#F97316"   # naranja — Val Loss
COLOR_POS   = "#22C55E"
COLOR_NEG   = "#EF4444"
BG_PLOT     = "#0F172A"
TEXT_PLOT   = "#E2E8F0"
GRID_PLOT   = "#1E293B"

# ── Estado global de sesión (modelo entrenado en memoria) ─────────────────────
SESION: dict = {
    "modelo":       None,   # SurrogateMLPMultiOutput
    "scaler_X":     None,   # StandardScaler para X
    "scaler_Y":     None,   # StandardScaler para Y
    "feat_cols":    [],     # nombres de columnas X, en orden
    "target_cols":  [],     # contaminantes Y detectados
    "parroquia":    "",
    "last_lags":    {},     # {col_lag: último_valor_válido}
    "feat_means":   {},     # {col: media_entrenamiento} — fallback inferencia
    "last_mes":     1,      # último mes del dataset (cíclica mes)
    "device":       "cpu",
    "arquitectura": "",
}


# ──────────────────────────────────────────────────────────────────────────────
# 2.  [FIX-4]  DETECCIÓN DE DISPOSITIVO (GPU / CPU)
# ──────────────────────────────────────────────────────────────────────────────

def _detectar_dispositivo() -> tuple[str, str]:
    """
    Detecta el mejor dispositivo disponible para PyTorch.
    Prioridad: CUDA (NVIDIA) → MPS (Apple) → CPU
    """
    if not TORCH_OK:
        return "cpu", "❌ PyTorch no instalado"
    try:
        if torch.cuda.is_available():
            name = torch.cuda.get_device_name(0)
            mem  = torch.cuda.get_device_properties(0).total_memory // (1024 ** 3)
            return "cuda", f"🟢 CUDA — {name} ({mem} GB)"
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return "mps", "🟢 MPS — Apple Silicon"
    except Exception:
        pass
    return "cpu", "🔵 CPU — sin GPU detectada"


DEVICE_STR, DEVICE_MSG = _detectar_dispositivo()
GPU_DISPONIBLE = DEVICE_STR in ("cuda", "mps")
log.info(f"Dispositivo DL: {DEVICE_MSG}")


def _badge_gpu_html() -> str:
    """Genera el badge HTML de hardware."""
    if GPU_DISPONIBLE:
        return f'<span class="gpu-badge gpu-on">🟢 {DEVICE_MSG}</span>'
    return f'<span class="gpu-badge gpu-off">🔵 {DEVICE_MSG}</span>'


# ──────────────────────────────────────────────────────────────────────────────
# 3.  EXPLORADOR DE ARCHIVOS CSV
# ──────────────────────────────────────────────────────────────────────────────

def listar_csvs(directorio: str = ".") -> list[str]:
    csvs = []
    for raiz, _, archivos in os.walk(directorio):
        for f in archivos:
            if f.lower().endswith(".csv"):
                csvs.append(os.path.relpath(os.path.join(raiz, f), directorio))
    csvs.sort()
    return csvs if csvs else ["(No se encontraron archivos CSV)"]


def refrescar_dropdown() -> gr.Dropdown:
    opciones = listar_csvs()
    return gr.Dropdown(choices=opciones, value=opciones[0] if opciones else None)


# ──────────────────────────────────────────────────────────────────────────────
# 4.  CARGA DE CSV  [FIX-1]
# ──────────────────────────────────────────────────────────────────────────────

def _cargar_csv(ruta: str) -> pd.DataFrame:
    """[FIX-1] Lee el CSV ignorando líneas decorativas con '#'."""
    try:
        return pd.read_csv(ruta, comment="#", low_memory=False, on_bad_lines="warn")
    except Exception as e:
        raise RuntimeError(f"Error al leer '{ruta}': {e}") from e


def _detectar_timestamp(df: pd.DataFrame) -> str | None:
    """Detecta la columna de tiempo por nombre o por parseo exitoso."""
    keywords = ("time", "fecha", "date", "hora", "datetime", "timestamp")
    candidatos = [c for c in df.columns if any(k in c.lower() for k in keywords)]
    if candidatos:
        return candidatos[0]
    for c in df.columns:
        try:
            pd.to_datetime(df[c].dropna().astype(str).iloc[:10], infer_datetime_format=True)
            return c
        except Exception:
            continue
    return None


# ──────────────────────────────────────────────────────────────────────────────
# 5.  [DL-1]  DETECCIÓN DINÁMICA DE ESQUEMA X / Y
# ──────────────────────────────────────────────────────────────────────────────

def _detectar_esquema_xy(df: pd.DataFrame) -> tuple[list[str], list[str], str]:
    """
    [DL-1] Inspecciona las cabeceras y separa automáticamente:

    Y (objetivo)  : columnas cuyo nombre EXACTO está en CONTAMINANTES_Y.
                    Regla de Oro: ninguna variable ajena puede estar en Y.
    X (entrada)   : todas las columnas numéricas restantes, excluyendo
                    las columnas de control (Fecha, Parroquia…).

    Returns
    -------
    x_cols  : list[str]  — features de entrada (en orden del CSV)
    y_cols  : list[str]  — contaminantes objetivo presentes en el dataset
    msg     : str        — descripción del esquema detectado
    """
    num_cols = set(df.select_dtypes(include=[np.number]).columns)
    all_cols = df.columns.tolist()

    # Y: solo contaminantes reconocidos, exactamente
    y_cols = [c for c in CONTAMINANTES_Y if c in all_cols]
    y_set  = set(y_cols)

    # X: numéricas, no en Y, no en columnas de control
    x_cols = [
        c for c in all_cols
        if c in num_cols
        and c not in y_set
        and c not in VARS_CONTROL
    ]

    n_y  = len(y_cols)
    n_x  = len(x_cols)
    miss = [c for c in CONTAMINANTES_Y if c not in all_cols]

    msg = (
        f"✅ Esquema detectado — "
        f"**Y contaminantes ({n_y}/6):** `{y_cols}`  ·  "
        f"**X features:** `{n_x}` columnas"
    )
    if miss:
        msg += f"  ·  _(no encontrados: {miss})_"
    return x_cols, y_cols, msg


def obtener_esquema_csv(ruta: str) -> str:
    """Botón 'Detectar esquema': devuelve mensaje informativo del X/Y."""
    if not ruta or ruta.startswith("(No"):
        return "⚠️ Selecciona un archivo válido."
    try:
        df_head = pd.read_csv(ruta, comment="#", nrows=5, low_memory=False)
        ts = _detectar_timestamp(df_head)
        if ts:
            df_head = df_head.drop(columns=[ts], errors="ignore")
        _, _, msg = _detectar_esquema_xy(df_head)
        return msg
    except Exception as e:
        return f"❌ Error: {e}"


# ──────────────────────────────────────────────────────────────────────────────
# 6.  [DL-2]  IMPUTACIÓN TEMPORAL Y PREPROCESAMIENTO
# ──────────────────────────────────────────────────────────────────────────────

def _imputar_temporal(df: pd.DataFrame) -> pd.DataFrame:
    """
    [DL-2] Forward-Fill → Backward-Fill síncrono.
    Las redes neuronales se rompen ante NaN; esta imputación garantiza
    tensores numéricos completos preservando la continuidad temporal.
    """
    return df.ffill().bfill()


def _preprocesar_v6(
    df: pd.DataFrame,
    ts_col: str,
    excluir_pandemia: bool = True,
) -> pd.DataFrame:
    """
    Pipeline de preparación del DataFrame:
    1. Convierte y ordena por timestamp.
    2. [FIX-3] Excluye años de pandemia.
    3. Coerce columnas de texto a numérico.
    4. [DL-2] Imputación temporal ffill → bfill.
    """
    df = df.copy()
    df[ts_col] = pd.to_datetime(df[ts_col], infer_datetime_format=True, errors="coerce")
    df = df.dropna(subset=[ts_col]).set_index(ts_col).sort_index()

    if excluir_pandemia:
        n_antes = len(df)
        df = df[~df.index.year.isin(ANOS_PANDEMIA)]
        n_excl = n_antes - len(df)
        if n_excl:
            log.info(f"[FIX-3] {n_excl:,} registros pandemia {ANOS_PANDEMIA} excluidos.")

    # Forzar numérico en columnas de texto
    obj_cols = df.select_dtypes(include=["object", "string"]).columns
    if len(obj_cols):
        df[obj_cols] = df[obj_cols].apply(pd.to_numeric, errors="coerce")

    # [DL-2] Imputación temporal
    df = _imputar_temporal(df)
    return df


def _split_cronologico(
    X: np.ndarray, Y: np.ndarray, ratio: float = 0.80
) -> tuple:
    """División temporal estricta sin data leakage."""
    corte = int(len(X) * ratio)
    return X[:corte], X[corte:], Y[:corte], Y[corte:]


# ──────────────────────────────────────────────────────────────────────────────
# 7.  [DL-3]  ARQUITECTURA MLP MULTI-SALIDA (PYTORCH)
# ──────────────────────────────────────────────────────────────────────────────

if TORCH_OK:

    class SurrogateMLPMultiOutput(nn.Module):
        """
        [DL-3] Red Neuronal Tabular Multi-Salida para predicción simultánea
        de los 6 contaminantes REMMAQ.

        Diseño:
          Entrada  → capas Dense[BatchNorm→ReLU→Dropout] → Salida lineal
          Salida   = n_output neuronas con activación lineal (regresión)
          Init     = Kaiming Normal (compatible con ReLU)

        Optimizada para GPU Tesla V100 con batches grandes (512-1024).
        """

        def __init__(
            self,
            n_input:       int,
            n_output:      int  = 6,
            hidden_dims:   list = None,
            dropout_rates: list = None,
        ):
            super().__init__()
            if hidden_dims   is None: hidden_dims   = [512, 256, 128, 64]
            if dropout_rates is None: dropout_rates = [0.30, 0.25, 0.20, 0.15]

            layers   = []
            prev_dim = n_input
            for h_dim, dr in zip(hidden_dims, dropout_rates):
                layers += [
                    nn.Linear(prev_dim, h_dim),
                    nn.BatchNorm1d(h_dim),
                    nn.ReLU(inplace=True),
                    nn.Dropout(dr),
                ]
                prev_dim = h_dim
            layers.append(nn.Linear(prev_dim, n_output))   # salida lineal

            self.network = nn.Sequential(*layers)
            self._init_weights()

        def _init_weights(self):
            for m in self.modules():
                if isinstance(m, nn.Linear):
                    nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                    nn.init.zeros_(m.bias)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            return self.network(x)

        @property
        def n_params(self) -> int:
            return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ──────────────────────────────────────────────────────────────────────────────
# 8.  FUNCIÓN DE ENTRENAMIENTO MLP
# ──────────────────────────────────────────────────────────────────────────────

def _entrenar_mlp(
    X_train: np.ndarray, Y_train: np.ndarray,
    X_val:   np.ndarray, Y_val:   np.ndarray,
    hidden_dims:   list,
    dropout_rates: list,
    n_epochs:   int   = 200, #nuemro de epocas aqui se puede cambiar
    batch_size: int   = 512,
    lr:         float = 1e-3,
    patience:   int   = 25,
    device:     str   = "cpu",
    log_fn            = None,
) -> tuple:
    """
    Bucle de entrenamiento del MLP multi-salida.

    Monitoreo de pérdida:
      - MSE conjunto sobre los n_output contaminantes en cada época
      - train_loss y val_loss registrados para la curva de aprendizaje [DL-4]

    Técnicas de regularización:
      - Dropout en capas ocultas
      - Weight decay L2 (Adam)
      - Gradient clipping (max_norm=1.0)
      - ReduceLROnPlateau (factor=0.5, patience=10)
      - Early stopping (patience configurable)

    Returns
    -------
    model        : SurrogateMLPMultiOutput (pesos del mejor epoch)
    train_losses : list[float]
    val_losses   : list[float]
    best_epoch   : int
    """
    if log_fn is None:
        log_fn = log.info

    dev = torch.device(device)

    # Tensores
    X_tr_t = torch.FloatTensor(X_train).to(dev)
    Y_tr_t = torch.FloatTensor(Y_train).to(dev)
    X_va_t = torch.FloatTensor(X_val).to(dev)
    Y_va_t = torch.FloatTensor(Y_val).to(dev)

    # DataLoader — shuffle=True en entrenamiento
    ds_train = TensorDataset(X_tr_t, Y_tr_t)
    drop     = len(ds_train) > batch_size    # evitar batch de tamaño 1
    loader   = DataLoader(
        ds_train, batch_size=batch_size,
        shuffle=True, drop_last=drop, num_workers=0,
    )

    n_input  = X_train.shape[1]
    n_output = Y_train.shape[1]

    model = SurrogateMLPMultiOutput(
        n_input, n_output, hidden_dims, dropout_rates
    ).to(dev)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=10, min_lr=1e-6
    )
    criterion = nn.MSELoss()

    # Estado
    train_losses: list[float] = []
    val_losses:   list[float] = []
    best_val   = float("inf")
    best_state = None
    no_improve = 0
    best_epoch = 1

    arch_str = f"{n_input} → {hidden_dims} → {n_output}"
    log_fn(f"  Arquitectura: {arch_str}  |  Parámetros: {model.n_params:,}")
    log_fn(f"  Batches/época: {len(loader)}  |  Batch: {batch_size}  |  Device: {device}")

    for epoch in range(1, n_epochs + 1):
        # ── Entrenamiento ──────────────────────────────────────────────────
        model.train()
        running = 0.0
        for X_b, Y_b in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(X_b), Y_b)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            running += loss.item() * len(X_b)
        train_loss = running / len(X_tr_t)

        # ── Validación ─────────────────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_va_t), Y_va_t).item()

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        scheduler.step(val_loss)

        # Early stopping
        if val_loss < best_val - 1e-8:
            best_val   = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
            best_epoch = epoch
        else:
            no_improve += 1

        if epoch == 1 or epoch % 20 == 0:
            log_fn(
                f"  Época {epoch:4d}/{n_epochs}  |  "
                f"Train MSE: {train_loss:.5f}  |  "
                f"Val MSE: {val_loss:.5f}  |  "
                f"LR: {optimizer.param_groups[0]['lr']:.1e}"
            )

        if no_improve >= patience:
            log_fn(
                f"  ⏹️  Early stopping en época {epoch}.  "
                f"Mejor: época {best_epoch}  (Val MSE = {best_val:.5f})"
            )
            break

    # Restaurar pesos del mejor epoch
    if best_state is not None:
        model.load_state_dict({k: v.to(dev) for k, v in best_state.items()})

    return model, train_losses, val_losses, best_epoch


# ──────────────────────────────────────────────────────────────────────────────
# 9.  IMPORTANCIA DE VARIABLES  (Permutation Importance — PyTorch)
# ──────────────────────────────────────────────────────────────────────────────

def _perm_importance_torch(
    model,
    X_val:      np.ndarray,
    Y_val:      np.ndarray,
    feat_names: list[str],
    device:     str = "cpu",
    n_repeats:  int = 4,
) -> pd.DataFrame:
    """
    Importancia de variables por permutación para el MLP multi-salida.
    Mide el incremento en MSE global al permutarse aleatoriamente cada feature.
    Positivo = feature importante; negativo = aporta ruido.
    """
    dev  = torch.device(device)
    crit = nn.MSELoss()

    # Subsample para velocidad (max 5000 filas)
    if len(X_val) > 5000:
        idx  = np.random.choice(len(X_val), 5000, replace=False)
        Xs   = X_val[idx]
        Ys   = Y_val[idx]
    else:
        Xs, Ys = X_val, Y_val

    X_t = torch.FloatTensor(Xs).to(dev)
    Y_t = torch.FloatTensor(Ys).to(dev)

    model.eval()
    with torch.no_grad():
        base_loss = crit(model(X_t), Y_t).item()

    importances = []
    for i in range(X_t.shape[1]):
        deltas = []
        for _ in range(n_repeats):
            X_p        = X_t.clone()
            perm_idx   = torch.randperm(X_p.shape[0], device=dev)
            X_p[:, i]  = X_p[perm_idx, i]
            with torch.no_grad():
                deltas.append(crit(model(X_p), Y_t).item() - base_loss)
        importances.append(float(np.mean(deltas)))

    return (
        pd.DataFrame({
            "Feature":    feat_names,
            "Importance": importances,
            "Std":        [0.0] * len(feat_names),
        })
        .sort_values("Importance", ascending=False)
        .reset_index(drop=True)
    )


# ──────────────────────────────────────────────────────────────────────────────
# 10. FIGURAS
# ──────────────────────────────────────────────────────────────────────────────

def _estilo_ax(ax, titulo: str, xlabel: str = "", ylabel: str = ""):
    ax.set_title(titulo, fontsize=11, fontweight="bold", color=TEXT_PLOT, pad=12)
    if xlabel: ax.set_xlabel(xlabel, color=TEXT_PLOT, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, color=TEXT_PLOT, fontsize=10)
    ax.tick_params(colors=TEXT_PLOT, labelsize=9)
    ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)


def _fig_curva_aprendizaje(
    train_losses: list[float],
    val_losses:   list[float],
    best_epoch:   int,
    parroquia:    str = "",
    arquitectura: str = "",
) -> plt.Figure:
    """
    [DL-4] Curva de aprendizaje GLOBAL del MLP — una sola gráfica.

    ─ Línea continua   = Train Loss (MSE)
    ─ Línea discontinua = Validation Loss (MSE)
    ─ Línea vertical   = mejor época (early stopping)

    Eje X : Época (1 → N entrenadas)
    Eje Y : MSE Loss conjunto sobre los 6 contaminantes  (↓ mejor)
    """
    fig, ax = plt.subplots(figsize=(13, 5), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)

    x = np.arange(1, len(train_losses) + 1)

    # Curvas principales
    ax.plot(x, train_losses, color=COLOR_TRAIN, lw=2.2,
            label="Train Loss (MSE)", zorder=5)
    ax.plot(x, val_losses, color=COLOR_VAL, lw=2.2, linestyle="--",
            label="Validation Loss (MSE)", zorder=5)

    # Banda entre train y val (zona de generalización)
    ax.fill_between(x, train_losses, val_losses,
                    alpha=0.07, color=COLOR_VAL, zorder=2)

    # Marcador de mejor época
    if 1 <= best_epoch <= len(val_losses):
        bv = val_losses[best_epoch - 1]
        bt = train_losses[best_epoch - 1]
        ax.axvline(
            best_epoch, color="#F59E0B", lw=1.8, linestyle=":", alpha=0.90,
            zorder=6, label=f"Mejor época: {best_epoch}   (Val MSE = {bv:.5f})",
        )
        # Punto resaltado sobre la curva de val
        ax.scatter([best_epoch], [bv], color="#F59E0B", s=70, zorder=8, edgecolors="#FCD34D", lw=1.5)
        offset = max(1, len(x) // 40)
        ax.annotate(
            f"  ← époc. {best_epoch}",
            xy=(best_epoch, bv),
            xytext=(best_epoch + offset, bv),
            color="#F59E0B", fontsize=8.5, va="center", zorder=9,
        )

    # Línea de convergencia de la validación final
    ax.axhline(val_losses[-1], color=COLOR_VAL, lw=0.7, linestyle=":", alpha=0.35)

    # Estilo
    arch_label = (
        arquitectura.replace("Estándar", "").replace("Ligera", "").replace("Profunda", "")
        .strip().strip("()").strip()
    )
    titulo = (
        f"Evolución del Entrenamiento — {parroquia}  ·  "
        f"MLP {arch_label}  ·  Salida 6 Contaminantes"
    )
    _estilo_ax(ax, titulo, "Época", "MSE Loss  (↓ mejor)")
    ax.legend(framealpha=0.22, labelcolor=TEXT_PLOT, facecolor=BG_PLOT,
              edgecolor=GRID_PLOT, fontsize=10, loc="upper right")

    nota = (
        f"Épocas entrenadas: {len(train_losses)}  ·  "
        f"Mejor época: {best_epoch}  ·  "
        f"MSE final — Train: {train_losses[-1]:.5f}  /  Val: {val_losses[-1]:.5f}"
    )
    fig.text(0.012, 0.012, nota, fontsize=7.5, color="#64748B", ha="left", va="bottom")
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    return fig


def _fig_feature_importance(fi_df: pd.DataFrame, parroquia: str = "") -> plt.Figure:
    """Gráfica horizontal de Permutation Importance (top 25 features)."""
    fi  = fi_df.head(25)
    n   = len(fi)
    fig, ax = plt.subplots(figsize=(9, max(4, n * 0.52)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    colores = [COLOR_POS if v >= 0 else "#EF4444" for v in fi["Importance"]]
    ax.barh(fi["Feature"][::-1], fi["Importance"][::-1],
            color=colores[::-1], align="center", alpha=0.85, height=0.65)
    ax.axvline(0, color="#475569", lw=0.9, linestyle="--")
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)
    titulo = "Feature Importance — MLP Multi-Salida  (Permutation Δ MSE)"
    if parroquia:
        titulo = f"[{parroquia}]  {titulo}"
    _estilo_ax(ax, titulo, "Incremento en MSE al permutar (Δ)", "")
    plt.yticks(color=TEXT_PLOT, fontsize=9)
    plt.xticks(color=TEXT_PLOT, fontsize=9)
    plt.tight_layout()
    return fig


def _fig_heatmap(df_full: pd.DataFrame, y_cols: list[str], parroquia: str = "") -> plt.Figure | None:
    """Mapa de calor de Pearson — contaminantes Y resaltados en naranja."""
    num_df = df_full.select_dtypes(include=[np.number])
    if num_df.shape[1] < 2:
        return None
    corr = num_df.corr(method="pearson")
    n    = len(corr)
    fig, ax = plt.subplots(
        figsize=(max(8, n * 0.60), max(7, n * 0.55)), facecolor=BG_PLOT
    )
    ax.set_facecolor(BG_PLOT)
    mask = np.zeros_like(corr, dtype=bool)
    mask[np.triu_indices_from(mask, k=1)] = True
    sns.heatmap(
        corr, mask=mask,
        cmap=sns.diverging_palette(230, 20, as_cmap=True),
        vmin=-1, vmax=1, center=0,
        annot=True, fmt=".2f",
        annot_kws={"size": 7.5, "color": TEXT_PLOT},
        linewidths=0.4, linecolor=GRID_PLOT,
        square=True, ax=ax, cbar_kws={"shrink": 0.7},
    )
    for y_col in y_cols:
        if y_col in corr.columns:
            idx = list(corr.columns).index(y_col)
            ax.add_patch(plt.Rectangle((idx, 0), 1, n, fill=False,
                                        edgecolor="#F97316", lw=2.0, clip_on=False))
            ax.add_patch(plt.Rectangle((0, idx), n, 1, fill=False,
                                        edgecolor="#F97316", lw=2.0, clip_on=False))
    titulo = "Correlación de Pearson  ·  [naranja = contaminantes Y]"
    if parroquia:
        titulo = f"[{parroquia}]  {titulo}"
    ax.set_title(titulo, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=14)
    ax.tick_params(colors=TEXT_PLOT, labelsize=8)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    plt.setp(ax.get_yticklabels(), rotation=0)
    ax.collections[0].colorbar.ax.tick_params(colors=TEXT_PLOT, labelsize=8)
    plt.tight_layout()
    return fig


# ──────────────────────────────────────────────────────────────────────────────
# 11. PIPELINE PRINCIPAL  (callback del botón Entrenar)
# ──────────────────────────────────────────────────────────────────────────────

def entrenar(
    csv_local:        str,
    csv_upload,
    nombre_modelo:    str,
    arquitectura:     str,
    n_epochs:         int,
    train_ratio:      float,
    patience:         int,
    excluir_pandemia: bool,
):
    """
    Pipeline completo de entrenamiento del Surrogate DL v6.0.

    Retorna 5 valores (Gradio outputs):
        metricas_md, fig_lc, fig_fi, fig_hm, estado
    """
    logs: list[str] = []

    def info(m): log.info(m);    logs.append(f"✅ {m}")
    def warn(m): log.warning(m); logs.append(f"⚠️  {m}")
    def err(m):  log.error(m);   logs.append(f"❌ {m}")

    def _estado():
        return "**Registro:**  " + "  ·  ".join(logs[-20:])   # últimas 20 entradas

    # 5 outputs: metricas_md + 3 figuras + estado
    # Error: "msg" + None×3 + estado = 5  ✓
    VACIO = (None, None, None)

    # Verificar PyTorch
    if not TORCH_OK:
        err("PyTorch no instalado.")
        return (
            "### ❌ PyTorch requerido\n\n"
            "Instala con:\n```bash\n"
            "pip install torch\n"
            "# GPU CUDA 11.8 (Tesla V100):\n"
            "pip install torch --index-url https://download.pytorch.org/whl/cu118\n```",
            *VACIO, _estado()
        )

    try:
        # ── 11.1  Resolver ruta del archivo ──────────────────────────────────
        if csv_upload is not None:
            ruta_csv = csv_upload if isinstance(csv_upload, str) else csv_upload.name
            info(f"Archivo subido: {Path(ruta_csv).name}")
        elif csv_local and not csv_local.startswith("(No"):
            ruta_csv = csv_local
            info(f"Archivo local: {ruta_csv}")
        else:
            err("Sin archivo de entrada.")
            return "### ❌ No se seleccionó ningún archivo.", *VACIO, _estado()

        if not Path(ruta_csv).exists():
            err(f"Archivo no encontrado: {ruta_csv}")
            return f"### ❌ Archivo no encontrado: `{ruta_csv}`", *VACIO, _estado()

        nombre_modelo = nombre_modelo.strip() or "surrogate_dl"
        parroquia     = Path(ruta_csv).stem.replace("_", " ").title()

        # ── 11.2  Cargar CSV ──────────────────────────────────────────────────
        df_raw = _cargar_csv(ruta_csv)
        info(f"CSV cargado: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")

        ts_col = _detectar_timestamp(df_raw)
        if ts_col is None:
            err("No se detectó columna de tiempo.")
            return "### ❌ Sin columna Timestamp/Date/Fecha.", *VACIO, _estado()
        info(f"Timestamp: '{ts_col}'")

        # ── 11.3  [DL-1] Detección de esquema X / Y ───────────────────────────
        df_schema = df_raw.drop(columns=[ts_col], errors="ignore")
        x_cols, y_cols, schema_msg = _detectar_esquema_xy(df_schema)
        info(schema_msg)

        if not y_cols:
            err("Sin contaminantes Y en el dataset.")
            return (
                f"### ❌ Sin variables objetivo\n\n{schema_msg}\n\n"
                f"Las columnas deben tener los nombres exactos: `{CONTAMINANTES_Y}`",
                *VACIO, _estado()
            )
        if not x_cols:
            err("Sin features X en el dataset.")
            return f"### ❌ Sin features de entrada\n\n{schema_msg}", *VACIO, _estado()

        # ── 11.4  [DL-2] Preprocesamiento + Imputación temporal ──────────────
        df = _preprocesar_v6(df_raw, ts_col, excluir_pandemia)
        if excluir_pandemia:
            info(f"Pandemia excluida: {ANOS_PANDEMIA}")

        # Separar matrices
        df_x = df[[c for c in x_cols if c in df.columns]].copy()
        df_y = df[[c for c in y_cols if c in df.columns]].copy()

        # Columnas presentes tras preprocesamiento
        x_cols_final = df_x.columns.tolist()
        y_cols_final = df_y.columns.tolist()

        # Eliminar filas donde todos los Y son NaN
        mask_y = df_y.notna().any(axis=1)
        df_x   = df_x[mask_y]
        df_y   = df_y[mask_y]

        # Rellenar NaN residuales en Y con la mediana
        for col in y_cols_final:
            df_y[col] = df_y[col].fillna(df_y[col].median())

        n_rows = len(df_x)
        info(f"Muestras limpias tras imputación: {n_rows:,}")

        if n_rows < 200:
            err(f"Datos insuficientes: {n_rows} filas (mínimo 200).")
            return f"### ❌ Datos insuficientes ({n_rows} filas).", *VACIO, _estado()

        # ── 11.5  Split cronológico ───────────────────────────────────────────
        X_arr = df_x.values.astype(np.float32)
        Y_arr = df_y.values.astype(np.float32)
        X_train, X_val, Y_train, Y_val = _split_cronologico(X_arr, Y_arr, train_ratio)
        info(
            f"Split {int(train_ratio*100)}/{int((1-train_ratio)*100)}: "
            f"Train={len(X_train):,}  |  Val={len(X_val):,}"
        )

        # ── 11.6  Escalado StandardScaler (X e Y) ────────────────────────────
        scaler_X = StandardScaler()
        scaler_Y = StandardScaler()

        X_tr_sc = np.nan_to_num(scaler_X.fit_transform(X_train), nan=0.0)
        X_va_sc = np.nan_to_num(scaler_X.transform(X_val),        nan=0.0)
        Y_tr_sc = np.nan_to_num(scaler_Y.fit_transform(Y_train),  nan=0.0)
        Y_va_sc = np.nan_to_num(scaler_Y.transform(Y_val),         nan=0.0)
        info(f"StandardScaler aplicado — X({X_tr_sc.shape[1]} feat.)  Y({Y_tr_sc.shape[1]} salidas)")

        # ── 11.7  [DL-3] Entrenamiento del MLP ───────────────────────────────
        arq_cfg       = ARQUITECTURAS.get(arquitectura, ARQUITECTURAS["Estándar  [512·256·128·64]"])
        hidden_dims   = arq_cfg["hidden_dims"]
        dropout_rates = arq_cfg["dropout_rates"]

        # Batch size adaptativo al tamaño del dataset
        if   n_rows <  5_000: batch_size = 64
        elif n_rows < 20_000: batch_size = 256
        else:                 batch_size = 512

        info(f"Entrenando MLP  ·  {arquitectura}  ·  Device: {DEVICE_STR}  ·  Batch: {batch_size}")

        dl_logs = []
        def dl_log(m): log.info(m); dl_logs.append(m)

        modelo, train_losses, val_losses, best_epoch = _entrenar_mlp(
            X_tr_sc, Y_tr_sc, X_va_sc, Y_va_sc,
            hidden_dims=hidden_dims, dropout_rates=dropout_rates,
            n_epochs=n_epochs, batch_size=batch_size,
            lr=1e-3, patience=patience,
            device=DEVICE_STR, log_fn=dl_log,
        )
        for l in dl_logs:
            logs.append(l)
        info(f"Entrenamiento completado — mejor época: {best_epoch}/{len(train_losses)}")

        # ── 11.8  Métricas por contaminante en validación ─────────────────────
        modelo.eval()
        dev = torch.device(DEVICE_STR)
        with torch.no_grad():
            Y_pred_sc = modelo(torch.FloatTensor(X_va_sc).to(dev)).cpu().numpy()

        Y_pred_orig = scaler_Y.inverse_transform(Y_pred_sc)
        Y_val_orig  = scaler_Y.inverse_transform(Y_va_sc)

        metricas_contam: list[dict] = []
        for i, col in enumerate(y_cols_final):
            yt = Y_val_orig[:, i]
            yp = np.clip(Y_pred_orig[:, i], 0, None)
            metricas_contam.append({
                "Contaminante": col,
                "MAE":  mean_absolute_error(yt, yp),
                "RMSE": float(np.sqrt(mean_squared_error(yt, yp))),
                "R2":   r2_score(yt, yp),
            })
        info("Métricas por contaminante calculadas.")

        # ── 11.9  Permutation Importance ─────────────────────────────────────
        info("Calculando Permutation Importance…")
        fi_df = _perm_importance_torch(
            modelo, X_va_sc, Y_va_sc, x_cols_final, DEVICE_STR, n_repeats=4
        )
        info("Feature Importance lista.")

        # ── 11.10  [DL-5] Lags históricos para inferencia ─────────────────────
        lag_cols  = [c for c in x_cols_final if "_lag_" in c.lower()]
        last_lags = {}
        for col in lag_cols:
            serie = df_x[col].dropna()
            last_lags[col] = float(serie.iloc[-1]) if len(serie) else 0.0

        feat_means = {col: float(df_x[col].mean()) for col in x_cols_final}
        last_mes   = int(df.index[-1].month) if hasattr(df.index[-1], "month") else 1

        # ── 11.11  Guardar modelo + artefactos ────────────────────────────────
        tag      = f"{nombre_modelo}_{parroquia.replace(' ', '_')}"
        pth_path = OUTPUT_DIR / f"{tag}_deep_learning_model.pth"
        torch.save({
            "model_state":   modelo.state_dict(),
            "n_input":       X_tr_sc.shape[1],
            "n_output":      Y_tr_sc.shape[1],
            "hidden_dims":   hidden_dims,
            "dropout_rates": dropout_rates,
            "scaler_X":      scaler_X,
            "scaler_Y":      scaler_Y,
            "feat_cols":     x_cols_final,
            "target_cols":   y_cols_final,
            "parroquia":     parroquia,
            "last_lags":     last_lags,
            "feat_means":    feat_means,
            "last_mes":      last_mes,
            "train_losses":  train_losses,
            "val_losses":    val_losses,
            "best_epoch":    best_epoch,
        }, pth_path)
        info(f"Modelo guardado: {pth_path.name}")

        fi_df.to_csv(OUTPUT_DIR / f"{tag}_feature_importance.csv", index=False)

        # ── 11.12  Actualizar sesión global ───────────────────────────────────
        SESION.update({
            "modelo":       modelo,
            "scaler_X":     scaler_X,
            "scaler_Y":     scaler_Y,
            "feat_cols":    x_cols_final,
            "target_cols":  y_cols_final,
            "parroquia":    parroquia,
            "last_lags":    last_lags,
            "feat_means":   feat_means,
            "last_mes":     last_mes,
            "device":       DEVICE_STR,
            "arquitectura": arquitectura,
        })
        info("Sesión actualizada → pestaña Inferencia lista.")

        # ── 11.13  Markdown de métricas ───────────────────────────────────────
        hw_label  = f"{'🟢 GPU' if GPU_DISPONIBLE else '🔵 CPU'}  —  {DEVICE_MSG}"
        n_params  = modelo.n_params
        arch_str  = " → ".join(
            str(d) for d in [X_tr_sc.shape[1]] + hidden_dims + [Y_tr_sc.shape[1]]
        )
        bv_loss   = val_losses[best_epoch - 1]
        bt_loss   = train_losses[best_epoch - 1]

        filas_m   = []
        for m in metricas_contam:
            ico = "🟢" if m["R2"] >= 0.7 else ("🟡" if m["R2"] >= 0.5 else "🔴")
            filas_m.append(
                f"| **{m['Contaminante']}** "
                f"| `{m['MAE']:.3f}` "
                f"| `{m['RMSE']:.3f}` "
                f"| {ico} `{m['R2']:.3f}` |"
            )

        metricas_md = f"""
## 🧠 Surrogate Deep Learning — *{parroquia}*

### Arquitectura MLP Multi-Salida

| Parámetro | Valor |
|-----------|-------|
| Arquitectura | `{arch_str}` |
| Parámetros entrenables | `{n_params:,}` |
| Features X | `{len(x_cols_final)}` columnas |
| Salidas Y | `{y_cols_final}` |
| Hardware | {hw_label} |
| Épocas entrenadas | `{len(train_losses)}` / `{n_epochs}` máx |
| Mejor época (early stop.) | `{best_epoch}` |
| Train MSE @ mejor época | `{bt_loss:.6f}` |
| Val MSE  @ mejor época | `{bv_loss:.6f}` |
| Modelo guardado | `{pth_path.name}` |

---

### Métricas por Contaminante — Validación ({int((1-train_ratio)*100)}% final)

| Contaminante | MAE | RMSE | R² |
|:---:|:---:|:---:|:---:|
{chr(10).join(filas_m)}

> Split cronológico estricto (sin data leakage).
> El **MAE y RMSE** están en las unidades originales del contaminante (µg/m³).
> Ver **📉 Curva de Aprendizaje** para la evolución del MSE global por época.
"""

        # ── 11.14  Figuras ────────────────────────────────────────────────────
        fig_lc = _fig_curva_aprendizaje(
            train_losses, val_losses, best_epoch, parroquia, arquitectura
        )
        fig_fi = _fig_feature_importance(fi_df, parroquia)
        df_hm  = pd.concat([df_x, df_y], axis=1).iloc[:20_000]
        fig_hm = _fig_heatmap(df_hm, y_cols_final, parroquia)
        info("Figuras generadas.")

        # 5 valores de retorno ✓
        return metricas_md, fig_lc, fig_fi, fig_hm, _estado()

    except ImportError as e:
        err(str(e))
        pkg = str(e).split("'")[-2] if "'" in str(e) else str(e).split()[-1]
        return (
            f"### ❌ Librería faltante\n```\n{e}\n```\n`pip install {pkg}`",
            *VACIO, _estado()
        )
    except Exception as e:
        err(str(e))
        return (
            f"### ❌ Error inesperado\n```\n{traceback.format_exc()}\n```",
            *VACIO, _estado()
        )


# ──────────────────────────────────────────────────────────────────────────────
# 12. [DL-5]  INFERENCIA LIMPIA MULTI-CONTAMINANTE
# ──────────────────────────────────────────────────────────────────────────────

def _clasificar_aqi(contaminante: str, valor: float) -> str:
    """Devuelve la etiqueta AQI para un contaminante y su valor."""
    for umbral, etiqueta in AQI_UMBRALES.get(contaminante, [(float("inf"), "⚪ N/D")]):
        if valor <= umbral:
            return etiqueta
    return "🟣 Peligroso"


def predecir_desde_sesion(
    temperatura:   float,
    humedad:       float,
    viento_vel:    float,
    viento_dir:    float,
    precipitacion: float,
    hora:          int,
) -> str:
    """
    [DL-5] Inferencia limpia: el usuario solo ingresa condiciones climáticas
    básicas y la hora. El backend realiza automáticamente:

    1. Codificación cíclica de hora y mes de referencia.
    2. Inyección de los últimos lags históricos del CSV de entrenamiento.
    3. Relleno de features desconocidas con la media del entrenamiento.
    4. Escalado → Predicción → Des-escalado.
    5. Salida: tabla con los 6 contaminantes + índice AQI.
    """
    if SESION["modelo"] is None:
        return (
            "### ⚠️ Modelo no disponible\n"
            "Entrena primero un modelo en la pestaña principal."
        )

    try:
        hora = int(hora)
        mes  = SESION.get("last_mes", 1)

        # ── 1. Features cíclicas ───────────────────────────────────────────
        hora_sin = float(np.sin(2 * np.pi * hora / 24))
        hora_cos = float(np.cos(2 * np.pi * hora / 24))
        mes_sin  = float(np.sin(2 * np.pi * mes  / 12))
        mes_cos  = float(np.cos(2 * np.pi * mes  / 12))

        # ── 2. Mapa de valores conocidos (clima + cíclicas + lags) ─────────
        vals = {
            "Temperatura":      float(temperatura),
            "Humedad":          float(humedad),
            "Viento_Velocidad": float(viento_vel),
            "Viento_Direccion": float(viento_dir),
            "Precipitacion":    float(precipitacion),
            "hora_sin":         hora_sin,
            "hora_cos":         hora_cos,
            "mes_sin":          mes_sin,
            "mes_cos":          mes_cos,
        }
        vals.update(SESION.get("last_lags", {}))   # inyectar lags históricos
        feat_means = SESION.get("feat_means", {})

        # ── 3. Vector X en el orden exacto del entrenamiento ──────────────
        row = [
            vals.get(col, feat_means.get(col, 0.0))
            for col in SESION["feat_cols"]
        ]

        # ── 4. Escalar → Predecir → Des-escalar ───────────────────────────
        X_sc = np.nan_to_num(
            SESION["scaler_X"].transform(np.array([row], dtype=np.float32)), nan=0.0
        )
        dev    = torch.device(SESION["device"])
        tensor = torch.FloatTensor(X_sc).to(dev)
        SESION["modelo"].eval()
        with torch.no_grad():
            pred_sc = SESION["modelo"](tensor).cpu().numpy()
        pred_orig = SESION["scaler_Y"].inverse_transform(pred_sc)[0]

        # ── 5. Tabla estética de resultados ────────────────────────────────
        target_cols = SESION["target_cols"]
        filas = []
        for i, contam in enumerate(target_cols):
            valor = float(max(0.0, pred_orig[i]))
            aqi   = _clasificar_aqi(contam, valor)
            ico   = aqi.split()[0]
            texto = " ".join(aqi.split()[1:])
            filas.append(
                f"| {ico} **{contam}** | `{valor:.3f} µg/m³` | {texto} |"
            )
        tabla = "\n".join(filas)

        arch_label = SESION.get("arquitectura", "MLP").split("[")[0].strip()

        return (
            f"## 🔮 Predicción Multi-Contaminante — *{SESION['parroquia']}*\n\n"
            f"| Contaminante | Concentración estimada | Calidad del Aire |\n"
            f"|:---:|:---:|:---:|\n"
            f"{tabla}\n\n"
            f"> **{arch_label}**  ·  "
            f"Hora de consulta: **{hora:02d}:00**  ·  "
            f"Mes de referencia: **{mes}**  ·  "
            f"Lags inyectados automáticamente desde el último registro del dataset."
        )

    except Exception:
        return f"### ❌ Error en predicción\n```\n{traceback.format_exc()}\n```"


# ──────────────────────────────────────────────────────────────────────────────
# 13. CSS
# ──────────────────────────────────────────────────────────────────────────────

CSS = """
:root {
    --bg:      #0F172A;
    --card:    #1E293B;
    --input:   #0D1525;
    --border:  #334155;
    --accent:  #6366F1;
    --accentH: #818CF8;
    --text:    #F1F5F9;
    --muted:   #94A3B8;
    --ok:      #22C55E;
    --warn:    #F59E0B;
    --err:     #EF4444;
    --r:       10px;
}
body, .gradio-container {
    background: var(--bg) !important;
    color: var(--text) !important;
    font-family: 'Inter', 'Segoe UI', sans-serif !important;
}
.gr-group, .gr-box {
    background: var(--card) !important;
    border: 1px solid var(--border) !important;
    border-radius: var(--r) !important;
    padding: 16px !important;
}
input, textarea, select {
    background: var(--input) !important;
    color: var(--text) !important;
    border: 1px solid var(--border) !important;
    border-radius: 6px !important;
}
label, .gr-label {
    color: var(--muted) !important;
    font-size: .8rem !important;
    font-weight: 700 !important;
    text-transform: uppercase;
    letter-spacing: .05em !important;
}
button.primary {
    background: linear-gradient(135deg, var(--accent), #7C3AED) !important;
    color: #fff !important; border: none !important;
    border-radius: 8px !important;
    font-weight: 800 !important; font-size: 1rem !important;
    padding: 12px 28px !important;
    box-shadow: 0 4px 20px rgba(99,102,241,.45);
    transition: all .15s;
}
button.primary:hover {
    transform: translateY(-2px);
    box-shadow: 0 6px 28px rgba(99,102,241,.6);
}
button.secondary {
    background: var(--card) !important;
    color: var(--text) !important;
    border: 1px solid var(--border) !important;
    border-radius: 8px !important;
}
.gr-markdown { color: var(--text) !important; }
.gr-markdown table { border-collapse: collapse; width: 100%; }
.gr-markdown th {
    background: #1E293B; color: var(--accentH);
    padding: 8px 14px; border: 1px solid var(--border);
}
.gr-markdown td {
    color: var(--text); padding: 7px 14px;
    border: 1px solid var(--border);
}
.gr-markdown tr:nth-child(even) td { background: #19253a; }
.gr-file {
    border: 2px dashed var(--accent) !important;
    border-radius: var(--r) !important;
}
.hero { text-align: center; padding: 24px 0 6px; }
.hero h1 {
    font-size: 2rem; font-weight: 900;
    background: linear-gradient(90deg, #6366F1, #38BDF8);
    -webkit-background-clip: text; -webkit-text-fill-color: transparent;
}
.hero p { color: var(--muted); font-size: .88rem; }
.gpu-badge {
    display: inline-block; padding: 4px 12px;
    border-radius: 20px; font-size: .75rem;
    font-weight: 700; margin-top: 4px;
}
.gpu-on  { background: rgba(34,197,94,.18);  color: #4ADE80; border: 1px solid #22C55E; }
.gpu-off { background: rgba(99,102,241,.15); color: #A5B4FC; border: 1px solid #6366F1; }
.logs-box {
    background: #0B1527 !important;
    border: 1px solid #1D3557 !important;
    border-radius: 8px; padding: 10px 14px;
    font-family: 'JetBrains Mono', monospace;
    font-size: .76rem; color: #7DD3FC;
    max-height: 130px; overflow-y: auto; line-height: 1.6;
}
.schema-box {
    background: #0D1A2D !important;
    border: 1px solid #1E3A5F !important;
    border-radius: 8px; padding: 10px 14px;
    font-size: .82rem; color: #93C5FD;
}
"""


# ──────────────────────────────────────────────────────────────────────────────
# 14. CONSTRUCCIÓN DE LA INTERFAZ GRADIO
# ──────────────────────────────────────────────────────────────────────────────

def construir_app() -> gr.Blocks:

    badge_html = _badge_gpu_html()

    with gr.Blocks(
        theme=gr.themes.Base(
            primary_hue="indigo",
            secondary_hue="sky",
            neutral_hue="slate",
            font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif"],
        ),
        css=CSS,
        title="Surrogate DL — Calidad del Aire v6",
    ) as app:

        # ── Hero ──────────────────────────────────────────────────────────────
        gr.HTML(f"""
        <div class="hero">
            <h1>🌬️ Surrogate Model — Calidad del Aire</h1>
            <p>Deep Learning Multi-Salida · MLP Tabular · 6 Contaminantes REMMAQ · v6.0</p>
            {badge_html}
        </div>
        """)

        with gr.Row(equal_height=False):

            # ══════════════════════════════════════════════
            # PANEL IZQUIERDO — CONTROLES
            # ══════════════════════════════════════════════
            with gr.Column(scale=1, min_width=360):

                # ── Dataset ──────────────────────────────────────────────────
                with gr.Group():
                    gr.Markdown("### 📂 Dataset")
                    with gr.Row():
                        csv_dropdown = gr.Dropdown(
                            label="CSVs detectados en el servidor",
                            choices=listar_csvs(), value=None,
                            interactive=True, scale=5,
                        )
                        btn_refresh = gr.Button(
                            "🔄", scale=1, min_width=46, variant="secondary"
                        )
                    csv_upload = gr.File(
                        label="O arrastra / sube un CSV externo",
                        file_types=[".csv"], type="filepath",
                    )
                    btn_detectar = gr.Button(
                        "🔍 Detectar esquema X / Y del CSV",
                        variant="secondary", size="sm",
                    )
                    info_schema = gr.Markdown(
                        value=(
                            "_Pulsa 'Detectar esquema' para verificar la separación "
                            "automática de contaminantes Y y features X._"
                        ),
                        elem_classes=["schema-box"],
                    )

                gr.HTML("<div style='height:8px'/>")

                # ── Configuración DL ─────────────────────────────────────────
                with gr.Group():
                    gr.Markdown("### ⚙️ Configuración Deep Learning")

                    nombre_modelo_input = gr.Textbox(
                        label="Nombre del modelo (.pth)",
                        value="surrogate_dl",
                    )
                    arquitectura_radio = gr.Radio(
                        label="Arquitectura MLP",
                        choices=list(ARQUITECTURAS.keys()),
                        value="Estándar  [512·256·128·64]",
                        info="Ligera = menos parámetros · Profunda = más capacidad (más lento)",
                    )
                    excluir_pandemia_chk = gr.Checkbox(
                        label="🚫 Excluir datos de pandemia (2020–2021)",
                        value=True,
                        info="Elimina registros de 2020-2021 para condiciones normales.",
                    )
                    with gr.Row():
                        n_epochs_slider = gr.Slider(
                            label="Épocas máximas",
                            minimum=50, maximum=500, step=10, value=200, scale=3,
                        )
                        patience_slider = gr.Slider(
                            label="Patience (early stop.)",
                            minimum=10, maximum=60, step=5, value=25, scale=2,
                        )
                    train_ratio_slider = gr.Slider(
                        label="Proporción de entrenamiento",
                        minimum=0.6, maximum=0.95, step=0.05, value=0.80,
                    )

                gr.HTML("<div style='height:10px'/>")
                btn_train = gr.Button(
                    "🚀  Iniciar Entrenamiento DL", variant="primary", size="lg"
                )
                gr.HTML("<div style='height:6px'/>")
                gr.Markdown("##### 🖥️ Registro de ejecución")
                estado_output = gr.Markdown(
                    value="_Esperando ejecución…_",
                    elem_classes=["logs-box"],
                )

            # ══════════════════════════════════════════════
            # PANEL DERECHO — RESULTADOS (5 tabs)
            # ══════════════════════════════════════════════
            with gr.Column(scale=2, min_width=580):
                with gr.Tabs():

                    # ── Tab 1: Métricas ───────────────────────────────────────
                    with gr.Tab("📊 Métricas"):
                        metricas_output = gr.Markdown(
                            value="*Las métricas aparecerán aquí tras entrenar.*"
                        )

                    # ── Tab 2: Curva de Aprendizaje [DL-4] ────────────────────
                    with gr.Tab("📉 Curva de Aprendizaje"):
                        gr.Markdown(
                            "**Evolución del MSE global** por época de entrenamiento.  \n"
                            "Una sola gráfica — **dos líneas** por parroquia:\n\n"
                            "- Línea **continua** = Train Loss (MSE)\n"
                            "- Línea **discontinua** = Validation Loss (MSE)\n"
                            "- Línea **vertical** = mejor época (early stopping)\n\n"
                            "> La pérdida se calcula sobre los **6 contaminantes simultáneamente** "
                            "(MSE conjunto del bloque Y)."
                        )
                        fig_lc_output = gr.Plot(
                            label="Curva de Aprendizaje — MSE Loss vs Época"
                        )

                    # ── Tab 3: Feature Importance ─────────────────────────────
                    with gr.Tab("🔍 Feature Importance"):
                        gr.Markdown(
                            "**Importancia por permutación** del MLP multi-salida.  \n"
                            "Un valor positivo significa que permutar esa feature "
                            "incrementa el MSE global → la feature es importante."
                        )
                        fig_fi_output = gr.Plot(
                            label="Permutation Importance (Δ MSE global)"
                        )

                    # ── Tab 4: Correlaciones ──────────────────────────────────
                    with gr.Tab("📊 Correlaciones"):
                        gr.Markdown(
                            "**Correlación de Pearson** entre todas las variables.  \n"
                            "Las filas/columnas de los contaminantes **Y** aparecen "
                            "resaltadas en naranja."
                        )
                        fig_hm_output = gr.Plot(
                            label="Mapa de calor — Correlación de Pearson"
                        )

                    # ── Tab 5: Inferencia Multi-Contaminante [DL-5] ───────────
                    with gr.Tab("🔮 Inferencia Multi-Contaminante"):
                        gr.Markdown(
                            "Ingresa **solo las condiciones climáticas actuales** y la hora.  \n"
                            "El backend inyecta automáticamente las variables cíclicas "
                            "y los últimos lags históricos del dataset.  \n"
                            "El modelo devuelve los **6 contaminantes** simultáneamente."
                        )
                        sesion_info = gr.Markdown(
                            "_⚠️ Entrena primero un modelo para activar esta sección._"
                        )

                        with gr.Group():
                            gr.Markdown("##### 🌡️ Condiciones Climáticas")
                            with gr.Row():
                                inp_temp  = gr.Number(
                                    label="Temperatura (°C)",
                                    value=18.0, minimum=-15.0, maximum=50.0,
                                    info="Temperatura ambiente", scale=2,
                                )
                                inp_hum   = gr.Slider(
                                    label="Humedad Relativa (%)",
                                    minimum=0, maximum=100, step=1, value=70, scale=3,
                                )
                            with gr.Row():
                                inp_wvel  = gr.Number(
                                    label="Velocidad del Viento (m/s)",
                                    value=2.5, minimum=0.0, maximum=30.0, scale=2,
                                )
                                inp_wdir  = gr.Number(
                                    label="Dirección del Viento (°)",
                                    value=180.0, minimum=0.0, maximum=360.0, scale=2,
                                )
                            with gr.Row():
                                inp_prec  = gr.Number(
                                    label="Precipitación (mm)",
                                    value=0.0, minimum=0.0, maximum=200.0, scale=2,
                                )
                                inp_hora  = gr.Slider(
                                    label="Hora del día (0–23 h)",
                                    minimum=0, maximum=23, step=1, value=12, scale=3,
                                )

                        gr.HTML("<div style='height:6px'/>")
                        btn_predecir  = gr.Button(
                            "🔮  Predecir los 6 Contaminantes", variant="primary"
                        )
                        resultado_pred = gr.Markdown(
                            value="_El resultado aparecerá aquí._"
                        )

        # ── Footer ────────────────────────────────────────────────────────────
        gr.HTML("""
        <div style="text-align:center;padding:18px 0 8px;color:#475569;font-size:.76rem;">
            Surrogate Model v6.0  ·  Deep Learning MLP Multi-Salida  ·
            PyTorch CUDA  ·  6 Contaminantes REMMAQ  ·  Gradio
        </div>
        """)

        # ── Eventos ───────────────────────────────────────────────────────────

        btn_refresh.click(fn=refrescar_dropdown, inputs=[], outputs=[csv_dropdown])

        def _wrapper_detectar(csv_local, csv_up):
            ruta = (
                (csv_up if isinstance(csv_up, str) else (csv_up.name if csv_up else None))
                or (csv_local if csv_local and not csv_local.startswith("(No") else None)
            )
            if not ruta:
                return "⚠️ Selecciona un archivo primero."
            return obtener_esquema_csv(ruta)

        btn_detectar.click(
            fn=_wrapper_detectar,
            inputs=[csv_dropdown, csv_upload],
            outputs=[info_schema],
        )

        # Actualizar info de sesión tras entrenar
        def _refresh_sesion_info():
            tgt = SESION.get("target_cols", [])
            xft = SESION.get("feat_cols",   [])
            par = SESION.get("parroquia",   "")
            arq = SESION.get("arquitectura","MLP").split("[")[0].strip()
            if tgt:
                return (
                    f"✅ Modelo en sesión: **{arq}** · "
                    f"Y = `{tgt}` · {len(xft)} features X · *{par}*"
                )
            return "_⚠️ Entrena primero un modelo._"

        # Entrenamiento → 5 outputs
        btn_train.click(
            fn=entrenar,
            inputs=[
                csv_dropdown, csv_upload,
                nombre_modelo_input, arquitectura_radio,
                n_epochs_slider, train_ratio_slider,
                patience_slider, excluir_pandemia_chk,
            ],
            outputs=[
                metricas_output,   # 1
                fig_lc_output,     # 2 — [DL-4] curva de aprendizaje
                fig_fi_output,     # 3
                fig_hm_output,     # 4
                estado_output,     # 5
            ],
        ).then(
            fn=_refresh_sesion_info,
            inputs=[],
            outputs=[sesion_info],
        )

        # Inferencia [DL-5]
        btn_predecir.click(
            fn=predecir_desde_sesion,
            inputs=[inp_temp, inp_hum, inp_wvel, inp_wdir, inp_prec, inp_hora],
            outputs=[resultado_pred],
        )

    return app


# ──────────────────────────────────────────────────────────────────────────────
# 15. PUNTO DE ENTRADA — RESILIENTE EN JUPYTER / SERVIDOR REMOTO
# ──────────────────────────────────────────────────────────────────────────────

def _imprimir_ip_fallback() -> None:
    import socket
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        print(f"  🖥️  IP local   : http://{s.getsockname()[0]}:<PUERTO>")
        s.close()
    except Exception:
        pass
    try:
        ips = subprocess.check_output(["hostname", "-I"], text=True, timeout=4).strip()
        print(f"  🌐  IPs servidor: {ips}")
    except Exception:
        pass
    try:
        ip_pub = subprocess.check_output(
            ["curl", "-s", "--max-time", "5", "https://ipecho.net/plain"],
            text=True, timeout=7,
        ).strip()
        if ip_pub:
            print(f"  🌍  IP pública  : http://{ip_pub}:<PUERTO>")
    except Exception:
        pass


if __name__ == "__main__":
    app = construir_app()

    print("\n" + "═" * 64)
    print("  🌬️  SURROGATE MODEL — Calidad del Aire  v6.0")
    print("  🧠  Deep Learning MLP Multi-Salida · 6 Contaminantes")
    print("═" * 64)
    print(f"  PyTorch OK  : {TORCH_OK}")
    print(f"  Dispositivo : {DEVICE_MSG}")
    print("  Puerto      : automático (server_port=None)")
    print("═" * 64)
    print("\n⏳ Generando link de acceso externo… por favor espera.\n")

    try:
        app.launch(
            server_name="0.0.0.0",
            server_port=None,
            share=True,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )
    except OSError as port_err:
        print(f"\n⚠️  Error de puerto: {port_err}")
        print("🔄  Reintentando en puerto 7861…\n")
        app.launch(
            server_name="0.0.0.0",
            server_port=7861,
            share=True,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )
    except Exception as tunnel_err:
        print(f"\n⚠️  El túnel público falló: {tunnel_err}")
        print("🔄  Relanzando sin túnel — acceso por IP directa:\n")
        _imprimir_ip_fallback()
        app.launch(
            server_name="0.0.0.0",
            server_port=None,
            share=False,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )

16:59:14 [INFO] Dispositivo DL: 🟢 CUDA — Tesla V100-PCIE-16GB (15 GB)



════════════════════════════════════════════════════════════════
  🌬️  SURROGATE MODEL — Calidad del Aire  v6.0
  🧠  Deep Learning MLP Multi-Salida · 6 Contaminantes
════════════════════════════════════════════════════════════════
  PyTorch OK  : True
  Dispositivo : 🟢 CUDA — Tesla V100-PCIE-16GB (15 GB)
  Puerto      : automático (server_port=None)
════════════════════════════════════════════════════════════════

⏳ Generando link de acceso externo… por favor espera.



17:00:15 [INFO] Archivo subido: dataset_ml_carapungo.csv
17:00:15 [INFO] CSV cargado: 160,262 filas × 19 columnas
17:00:15 [INFO] Timestamp: 'Timestamp'
17:00:15 [INFO] ✅ Esquema detectado — **Y contaminantes (6/6):** `['PM25', 'PM10', 'O3', 'CO', 'NO2', 'SO2']`  ·  **X features:** `12` columnas
17:00:16 [INFO] Muestras limpias tras imputación: 160,262
17:00:16 [INFO] Split 80/19: Train=128,209  |  Val=32,053
17:00:16 [INFO] StandardScaler aplicado — X(12 feat.)  Y(6 salidas)
17:00:16 [INFO] Entrenando MLP  ·  Profunda  [1024·512·256·128·64]  ·  Device: cuda  ·  Batch: 512
17:00:18 [INFO]   Arquitectura: 12 → [1024, 512, 256, 128, 64] → 6  |  Parámetros: 714,950
17:00:18 [INFO]   Batches/época: 250  |  Batch: 512  |  Device: cuda
17:00:23 [INFO]   Época    1/200  |  Train MSE: 0.79171  |  Val MSE: 1.11314  |  LR: 1.0e-03
17:01:47 [INFO]   Época   20/200  |  Train MSE: 0.50143  |  Val MSE: 1.00768  |  LR: 1.0e-03
17:03:16 [INFO]   Época   40/200  |  Train MSE: 0.47431  |  Val MSE: 1.019